# The ISM prediction from the catalogues alone — the 100, 50 and 25 Mpc boxes

Figure H of `paper_figures_quenched_m25.ipynb` (Part 6, `paper_ism_prediction`) reads the ISM content an observer cannot cheaply measure ($M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$) off what they can (stellar age, $\Sigma_{\rm e}$, sSFR) for the quenched galaxies of the 25 Mpc box, with the model side taken from CIGALE fits of mock core photometry. That figure needs the whole m25 pipeline (particle cut-outs, RT, CIGALE). This notebook redraws it with **catalogue columns only** — every model quantity is a galaxy-level caesar column of the SIMBA boxes at the same anchor snapshots — so the larger boxes enter at no cost. The gas row of the grids shows, for the simulation, either $1.36\,M_{\rm H_2}/M_\star$ or the whole-galaxy gas fraction $M_{\rm gas}/M_\star$ (`SIM_GAS_ROW`, default `"fgas"`: every member gas particle, He included) against the $\alpha_{\rm CO}$ H$_2$ of the observations either way; the two catalogue quantities are not nested (caesar sums HI / H$_2$ over the halo gas assigned to the galaxy — `masses.HI` exceeds `masses.gas` — so the median $M_{\rm gas}/1.36\,M_{\rm H_2}$ is only +0.08 dex in cis100 / cis50, +0.26 in cis25), and `lfh2` stays the catalogue H$_2$ in every table:

| box | catalogue | gas particle mass | role here |
|---|---|---|---|
| `cis100` | m100n1024 | $1.8\times10^7$ M$_\odot$ | the statistics: hundreds of quenched galaxies per anchor |
| `cis50` | m50n512 | $1.8\times10^7$ M$_\odot$ (same resolution as m100) | an independent volume at the same resolution; its feedback variants `cis50nox` / `cis50noagn` (Parts 4–5) are the physics test |
| `cis25` | m25n512 | $2.3\times10^6$ M$_\odot$ | the box of the m25 sample under the *catalogue* definitions — the rung that links the catalogue tracks to figure H |

* **Selection** — the m25 Part 1 rule on the catalogue columns at the anchor: $\log M_\star > 10$, passive (sSFR $< 0.2/t_{\rm H}(z)$, instantaneous `sfr`), $\geq 21$ gas and $\geq 20$ star particles, $M_{\rm dust} \geq 10^{-4} M_{\rm H_2}$; the tracks under the figure's $\log M_\star > 10.25$. No AGN coupling class exists at catalogue level (it needs the histories); the tracks are the *all quenched* ones of figure H, per box, Part 3 splits them by the AGN state at the anchor (the jet criterion), and Part 6 rebuilds the m25 coupling classes proper from the catalogue histories — by default **two classes**, weak / strong, the rule's intermediate galaxies redistributed at $w_{\rm pre} = 0.30$ (`CLASS_SCHEME`, `TWO_CLASS_THR`; `"three"` restores the rule) — (merger trees + `find_quenching_times`).
* **Model quantities** — `masses.dust` / `masses.stellar`, $1.36\times$ `masses.H2` / `masses.stellar`, `ages.mass_weighted`, `sfr` / `masses.stellar`, $\Sigma_{\rm e} = M_\star / 2\pi R_{\rm e}^2$ with $R_{\rm e} = 0.75\times$ the 3-D stellar half-mass radius (`radii.stellar_half_mass`, comoving kpc $\to$ physical). Whole-galaxy (FOF) quantities, one row per galaxy: **no aperture anywhere in this notebook** — there are no particle files behind the 100 / 50 Mpc boxes, so the core / outskirt split of the m25 work does not exist here and every model quantity is the galaxy the catalogue sees.
* **Reference** — the m25 particle sample drawn behind the catalogue tracks *the way this notebook sees every box*: `M25_REF_MODEL = "catalogue"` takes the quenched galaxies of the m25 selection table (`powderday_quenched_selection_pt.fits`) from the cis25 catalogue rows — the same whole-galaxy columns, the m25 AGN classes — and bins them like figure H (all quenched + per class), so an offset between a box and the m25 sample is resolution / volume, not aperture. No particle rung reproduces the catalogue galaxy (checked on the 266 Q: the projected 0–10 kpc disc carries +0.26 dex more dust and +0.34 dex more gas than the FOF galaxy, the 0–100 kpc rung +1.3 dex), so the particle-based references are kept only as options: `"sim"` = figure H$_{\rm sim}$ (`paper_ism_prediction_sim.csv`, the SIMBA truth in the 0–10 / 0–32 kpc discs), `"cigale"` = figure H proper (`paper_ism_prediction.csv`, the CIGALE fits of the mock 0–3.2 kpc *core* photometry). The observed points are the same ALMA-C11 / Spilker+18 / ADF22-QG1 values (integrated: whole sources), coloured by $A_V$.
* **Outputs** — `output/box_resolution/ism_prediction/`: `ism_prediction_catalogue.fits` (the cache: every galaxy with $\log M_\star > 9.5$ at every (box, anchor) read), `paper_ism_prediction_boxes.{png,pdf,csv}` (+ `_points.csv`), `paper_ism_prediction_agnstate_<box>.{png,pdf,csv}`, `paper_ism_prediction_variants_m50.*` (Part 4), `paper_ism_prediction_agnstate_m50_<variant>.*`, `paper_ism_prediction_agnstate_variants_m50.*` + `_scorecard.csv` (Part 5), `ism_prediction_histories_<box>.hdf5`, `ism_prediction_agn_classes.csv`, `paper_ism_prediction_agnclass_<box>.*`, `paper_ism_prediction_agnclass_boxes.*` + `_scorecard.csv` (Part 6). Part 7: `paper_ism_prediction_dusty_tail.{png,pdf,csv}` — the individual galaxies behind the medians and the dusty tail per population. Part 8: `paper_ism_prediction_dusty_split{,_stats,_predictors}.csv`, `paper_ism_prediction_dusty_split_<box>.*`, `_summary.*` — the three dust sides (dust-rich $\geq 10^{-3.5}$, undetected, no-dust $< 10^{-5}$) and the class as a predictor of the side. Part 9: `ism_prediction_xray_exposure.csv` (per galaxy: $E_x$, duty, gas-poor time, timing, sequence), `paper_ism_prediction_xray_exposure.{png,pdf,csv}` + `_populations.csv`, `_predictors.csv`, `_stats.csv` — the X-ray channel after quenching (exposure, first-episode timing, quenching sequence) against the dust-to-gas ratio, the class held fixed.

Run anywhere the catalogues are visible (`SHARE`, default `/mnt/home/share/simbas`; a local run sets `SIMBA_SHARE` to the mount). Reads: the catalogues, `obs_data/almac11/{almac11_gas_dust,age_sersic_sigma}.csv`, `obs_data/literature/{av_re_literature,spilker18_legac}.csv`, optionally `output/cis25/plots/paper_m25_pt/paper_ism_prediction.csv`.

In [ ]:
# ── Part 0 — configuration: figure H (paper_figures_quenched_m25 Part 6) from the caesar catalogues alone; no particles, no RT, no CIGALE ──
import os
import time
import warnings
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patheffects import withStroke
from matplotlib.ticker import FuncFormatter
from astropy.cosmology import Planck15 as COSMO   # the cosmology of the m25 quenching machinery (t_H of the passive cut)
from astropy.table import Table
from scipy.stats import spearmanr, mannwhitneyu

SHARE = os.environ.get("SIMBA_SHARE", "/mnt/home/share/simbas")   # root of the caesar catalogues (a local run points it at the gvfs mount)
BOXES = {   # catalogue directory under SHARE, file pattern, legend label, line style (Okabe-Ito colours; cis25 / cis100 as in box_resolution_comparison)
    "cis100": dict(groups="SIMBA_100/Groups", fmt="m100n1024_{snap:03d}.hdf5", label="m100n1024 (100 Mpc/h)", color="#D55E00", ls="-", marker="o", m_gas=1.82e7),
    "cis50":  dict(groups="SIMBA_50/Groups",  fmt="m50n512_{snap:03d}.hdf5",  label="m50n512 (50 Mpc/h)",  color="#009E73", ls="-", marker="s", m_gas=1.82e7),
    "cis25":  dict(groups="SIMBA_25/Groups",  fmt="m25n512_{snap:03d}.hdf5",  label="m25n512 (25 Mpc/h)",  color="#0072B2", ls="-", marker="D", m_gas=2.28e6),
    # the 50 Mpc feedback variants (Part 4): same box, same resolution, one feedback channel removed at a time (ROE m50n512/<variant>/catalogs)
    "cis50nox":   dict(groups="SIMBA_50/s50nox/Groups",   fmt="m50n512_{snap:03d}.hdf5", label="m50n512 no X-ray feedback (s50nox)",     color="#CC79A7", ls="-", marker="^", m_gas=1.82e7),
    "cis50nojet": dict(groups="SIMBA_50/s50nojet/Groups", fmt="m50n512_{snap:03d}.hdf5", label="m50n512 no jet, no X-ray (s50nojet)",     color="#56B4E9", ls="-", marker="P", m_gas=1.82e7),
    "cis50noagn": dict(groups="SIMBA_50/s50noagn/Groups", fmt="m50n512_{snap:03d}.hdf5", label="m50n512 no AGN feedback (s50noagn)",      color="#E69F00", ls="-", marker="v", m_gas=1.82e7),
}
BOX_ORDER = ["cis100", "cis50", "cis25"]                  # Part 2: the resolution / volume ladder
VARIANTS  = ["cis50", "cis50nox", "cis50noagn"]           # Part 4: the 50 Mpc feedback variants drawn (cis50nojet is configured; add it here)
ALL_BOXES = list(dict.fromkeys(BOX_ORDER + VARIANTS))     # everything Part 1 reads
ANCHORS = {0.3: 134, 0.5: 125, 0.7: 116, 0.85: 110, 1.0: 105, 1.15: 100, 1.3: 95, 1.5: 90, 1.8: 83, 2.0: 78}   # m25 TARGET_REDSHIFTS -> snapshot (shared schedule)
ROE_CATALOG_URL = {"cis100": "http://simba.roe.ac.uk/simdata/m100n1024/s50/catalogs/m100n1024_{snap:03d}.hdf5",     # where a missing catalogue comes from
                   "cis50": "http://simba.roe.ac.uk/simdata/m50n512/s50/catalogs/m50n512_{snap:03d}.hdf5",            # (wget -c into SHARE/<groups>/)
                   "cis50nox": "http://simba.roe.ac.uk/simdata/m50n512/s50nox/catalogs/m50n512_{snap:03d}.hdf5",
                   "cis50nojet": "http://simba.roe.ac.uk/simdata/m50n512/s50nojet/catalogs/m50n512_{snap:03d}.hdf5",
                   "cis50noagn": "http://simba.roe.ac.uk/simdata/m50n512/s50noagn/catalogs/m50n512_{snap:03d}.hdf5"}

# ── the m25 Part 1 selection (powderday_flux_quenched_m25 cell 2), evaluated on the catalogue columns at the anchor ──
MASS_FLOOR     = 10.0     # log10(M*/Msun) > 10 (the pool)
PASSIVE_FACTOR = 0.2      # passive if sSFR < 0.2 / t_H(z) (instantaneous catalogue sfr over masses.stellar)
NGAS_MIN       = 21       # >= 21 gas particles at the anchor
NSTAR_MIN      = 20       # >= 20 star particles
DUST_TO_H2_MIN = 1e-4     # M_dust >= 1e-4 M_H2 (drops the dust-poor half of the passive pool, as in m25)
MASS_MIN       = 10.25    # the figure's cut on the tracks (P6_MASS_MIN)
POOL_LOGM      = 9.5      # galaxies read from a catalogue into the cache (every cut above is applied later, in memory)
CENTRALS_ONLY  = False    # m25 did not cut on central / satellite; True restricts the tracks to centrals
HE             = 1.36     # SIMBA H2 is hydrogen-only; the observed alpha_CO masses include helium
SIM_GAS_ROW    = "fgas"   # the simulation quantity on the H2 row of the grids: "fh2" = 1.36 x the catalogue H2 over M*, "fgas" = the whole-galaxy gas (every member
                          # gas particle, He included) over M*; the observed points stay the alpha_CO H2 masses either way. Not nested: the catalogue HI / H2 are summed
                          # over the halo gas assigned to the nearest galaxy (masses.HI > masses.gas), so M_gas / (1.36 M_H2) is only +0.08 dex (cis100 / cis50), +0.26 (cis25)
SFR_COL        = "sfr"    # sSFR of the tracks: "sfr" (instantaneous, = the selection) | "sfr_100" (100 Myr average)
MSTAR_COL      = "mstar"  # M* behind the fractions, sSFR and Sigma_e: "mstar" (whole galaxy) | "mstar_30" (30 comoving kpc sphere)
R_PROJ_OVER_3D = 0.75     # projected half-mass radius = this x the catalogue 3-D stellar half-mass radius (0.75: Hernquist-like profiles; 1.0 = none)
SSFR_FLOOR     = 1e-13    # sSFR below this (zero SFR) is drawn at the floor (log axis)
NBIN_X, NMIN_X = 6, 30    # tracks: at most NBIN_X equal-count x bins of >= NMIN_X galaxies (fewer than 2 x NMIN_X galaxies: 4 bins of >= 5)
NBOOT          = 500      # bootstrap draws of the bin medians (16–84 % band)
JET_LOGMBH, JET_FEDD = 7.5, 0.2   # Part 3 AGN-state proxy at the anchor: jet mode = log M_BH > 7.5 and f_Edd < 0.2 (SIMBA's jet criterion)
OVERWRITE_CACHE = False   # True re-reads every catalogue; else the cache is used and only the (box, anchor) pairs it lacks are read

OUT            = os.path.join(os.getcwd(), "output", "box_resolution", "ism_prediction")
CACHE_FITS     = os.path.join(OUT, "ism_prediction_catalogue.fits")                                                  # Part 1 cache
M25_REF_MODEL  = "catalogue"   # the m25 reference tracks drawn behind the catalogue ones — the m25 particle sample seen the way this notebook sees every box:
                               # "catalogue" = its quenched galaxies (powderday_quenched_selection_pt.fits, pop Q) READ FROM THE cis25 CATALOGUE: the same whole-galaxy
                               # columns, its own AGN classes (built in Part 2; no particle product needed) | "sim" = figure H_sim of the paper notebook
                               # (paper_ism_prediction_sim.csv: the SIMBA truth in the projected 0–10 kpc ISM / 0–32 kpc stellar discs — cylinders through the 100 kpc
                               # cut-out, +0.26 dex dust and +0.34 dex gas over the catalogue galaxy) | "cigale" = figure H proper (paper_ism_prediction.csv, the
                               # CIGALE fits of the mock 0–3.2 kpc CORE photometry — an aperture no catalogue quantity can match)
M25_REF_CFG    = {"catalogue": dict(csv=None, label="m25 sample (catalogue rows)",
                                    note="m25 reference: the quenched galaxies of the m25 particle sample read from the cis25 catalogue\n   (the same whole-galaxy columns as every box; the m25 AGN classes)"),
                  "sim":       dict(csv="paper_ism_prediction_sim.csv", label="figure H_sim (m25, SIMBA truth, 0–10 / 0–32 kpc discs)",
                                    note="figure H_sim reference: the SIMBA truth of the m25 particle sample in the projected 0–10 kpc (ISM) and\n   0–32 kpc (stars) discs — cylinders through the cut-out: +0.26 dex dust, +0.34 dex gas over the catalogue galaxy"),
                  "cigale":    dict(csv="paper_ism_prediction.csv", label="figure H (m25, CIGALE core fits)",
                                    note="figure H reference: CIGALE fits of the mock 0–3.2 kpc core photometry\n   of the m25 particle sample")}[M25_REF_MODEL]
M25_TRACKS_CSV = os.path.join(os.getcwd(), "output", "cis25", "plots", "paper_m25_pt", M25_REF_CFG["csv"]) if M25_REF_CFG["csv"] else None   # "sim" / "cigale"
M25_SELECTION  = os.path.join(os.getcwd(), "output", "cis25", "tables", "powderday_quenched_selection_pt.fits")   # the m25 particle sample (m25 Part 3): the "catalogue" reference, the cis25 class check of Part 6
M25_REF_LABEL, M25_REF_NOTE = M25_REF_CFG["label"], M25_REF_CFG["note"]
M25_REF_NBIN, M25_REF_NMIN = 4, 3   # "catalogue": the reference tracks binned like figure H (up to 4 equal-count bins of >= 3 galaxies)

# ── the AGN coupling classes (the m25 pre_threshold rule on w_pre; Part 6 rebuilds w_pre from the catalogue histories, the m25 selection table carries it for the reference) ──
PRE_THR_WEAK, PRE_THR_STRONG = 0.10, 0.50   # the m25 rule: weak below, strong at or above, intermediate between (powderday_flux_quenched_m25 cell "2")
CLASS_SCHEME  = "two"     # the classes used everywhere: "two" = weak / strong only, the rule's intermediate galaxies redistributed by w_pre at TWO_CLASS_THR |
                          # "three" = the rule as is (weak / intermediate / strong)
TWO_CLASS_THR = 0.30      # "two": w_pre below -> weak, at or above -> strong (the middle of the rule's intermediate band 0.10–0.50)
CLASSES  = ["weak", "strong"] if CLASS_SCHEME == "two" else ["weak", "intermediate", "strong"]
CLASSES3 = ["weak", "intermediate", "strong"]


def coupling_class(w):
    """A finite w_pre -> its class under CLASS_SCHEME ("three": the m25 rule; "two": one cut at TWO_CLASS_THR)."""
    if CLASS_SCHEME == "two":
        return "strong" if w >= TWO_CLASS_THR else "weak"
    return "strong" if w >= PRE_THR_STRONG else ("weak" if w < PRE_THR_WEAK else "intermediate")


def rule_class(w):
    """A finite w_pre -> the m25 rule's three-class label (the check against the m25 pipeline is always on these)."""
    return "strong" if w >= PRE_THR_STRONG else ("weak" if w < PRE_THR_WEAK else "intermediate")
OBS_GD_CSV     = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_gas_dust.csv")       # ALMA-C11 dust / CO + fiducial CIGALE age, M*, SFR, A_V
OBS_STRUCT_CSV = os.path.join(os.getcwd(), "obs_data", "almac11", "age_sersic_sigma.csv")       # ALMA-C11 R_e (COSMOS-Web / ACS), Sersic n
LIT_CSV        = os.path.join(os.getcwd(), "obs_data", "literature", "av_re_literature.csv")    # LEGA-C rows behind the Spilker+18 points
SPILKER_CSV    = os.path.join(os.getcwd(), "obs_data", "literature", "spilker18_legac.csv")     # Spilker+18 LEGA-C passive galaxies (M_H2)
os.makedirs(OUT, exist_ok=True)

PAPER_DPI = 300
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9, "xtick.labelsize": 9.5,
                     "ytick.labelsize": 9.5, "axes.titleweight": "bold", "pdf.fonttype": 42})
CLASS_COLOR = {"weak": "#1b9e77", "intermediate": "#e08214", "strong": "#a50f15"}   # the m25 AGN coupling classes (Part 6, the m25 reference)
C_OBS_EDGE  = "#3b0f2a"


def need(path, made_by):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: {made_by}")
    return path


def paper_save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{stem}.{ext}"), dpi=PAPER_DPI, bbox_inches="tight")
    print("figure ->", os.path.join(OUT, f"{stem}.{{png,pdf}}"))


def cat_path(box, snap):
    return os.path.join(SHARE, BOXES[box]["groups"], BOXES[box]["fmt"].format(snap=int(snap)))


AVAIL = {box: [(zt, s) for zt, s in ANCHORS.items() if os.path.exists(cat_path(box, s))] for box in ALL_BOXES}
print(f"catalogues under {SHARE}; outputs -> {OUT}")
for box in ALL_BOXES:
    miss = [s for zt, s in ANCHORS.items() if not os.path.exists(cat_path(box, s))]
    print(f"  {box:10s} {len(AVAIL[box]):2d} of {len(ANCHORS)} anchors: z = {', '.join(f'{zt:g}' for zt, s in AVAIL[box])}"
          + (f"; missing snapshots {miss}" + (f" (ROE: {ROE_CATALOG_URL[box].format(snap=miss[0])} ...)" if box in ROE_CATALOG_URL else "") if miss else ""))
    print(f"          gas particle mass {BOXES[box]['m_gas']:.2e} Msun -> the >= {NGAS_MIN} gas particle cut is a gas mass floor of {NGAS_MIN * BOXES[box]['m_gas']:.1e} Msun")
for _p, _by in [(OBS_GD_CSV, "git pull"), (OBS_STRUCT_CSV, "git pull"), (LIT_CSV, "git pull"), (SPILKER_CSV, "git pull"),
                (M25_SELECTION, "m25 Part 3 (optional: the m25 sample behind the catalogue reference / the Part 6 class check)")] + \
                ([(M25_TRACKS_CSV, f"paper_figures_quenched_m25 Part 6, model {M25_REF_MODEL!r} (optional: the m25 reference tracks)")] if M25_TRACKS_CSV else []):
    print(f"  {'ok     ' if os.path.exists(_p) else 'MISSING'} {os.path.relpath(_p, os.getcwd())}   [{_by}]")

## Part 1 — the catalogue rows and the m25 selection

`read_anchor` opens one caesar catalogue with h5py and keeps every galaxy above `POOL_LOGM` with the columns of `CAT_COLS` (masses, `sfr` / `sfr_100`, particle counts, `central`, the mass-weighted stellar age, the stellar and gas half-mass radii converted from comoving to physical kpc, the caesar $\kappa_{\rm rot}$ of stars and gas, $M_{\rm BH}$ and $f_{\rm Edd}$). The rows of every available (box, anchor) are cached in `CACHE_FITS`; a later run reads only the pairs the cache lacks (e.g. the 50 Mpc anchors once their catalogues are downloaded).

`m25_cuts` applies the Part 1 rule of the m25 notebook — massive, passive, $\geq 21$ gas / $\geq 20$ star particles, $M_{\rm dust} \geq 10^{-4} M_{\rm H_2}$ — and the funnel per (box, anchor) is printed the way the m25 stats table is. `Q` = the quenched sample with the derived axes (`fdust`, `fh2`, `fgas`, `ssfr`, `re`, `sig_e`, `age`, the jet flag), `QM` = the tracks' sample under `MASS_MIN`. Two things the funnel makes visible: the gas-particle cut is a **gas-mass floor 8 times higher** in the 100 / 50 Mpc boxes than in the 25 Mpc one (same particle count, coarser particles), and the passive pool of the large box is dominated by satellites at low redshift (`central` is carried; `CENTRALS_ONLY` restricts).

In [ ]:
# ── Part 1 — the catalogue rows of every available (box, anchor) -> CAT (cached); the m25 selection -> Q; the tracks' sample -> QM ──
CAT_COLS = {   # cache column: candidate datasets under galaxy_data (the first present wins; absent -> NaN)
    "mstar": ["dicts/masses.stellar"], "mstar_30": ["dicts/masses.stellar_30kpc", "dicts/masses.star_30kpc"],
    "sfr": ["sfr"], "sfr_100": ["sfr_100"], "mgas": ["dicts/masses.gas"], "mdust": ["dicts/masses.dust"], "mh2": ["dicts/masses.H2"], "mhi": ["dicts/masses.HI"],
    "ngas": ["ngas"], "nstar": ["nstar"], "central": ["central"], "age": ["dicts/ages.mass_weighted"],
    "r_half_star": ["dicts/radii.stellar_half_mass"], "r_half_gas": ["dicts/radii.gas_half_mass"],
    "kappa_star": ["dicts/rotation.stellar_kappa_rot"], "kappa_gas": ["dicts/rotation.gas_kappa_rot"],
    "mbh": ["dicts/masses.bh"], "fedd": ["bh_fedd"]}
COMOVING_KPC = ["r_half_star", "r_half_gas"]   # caesar radii are comoving kpc (kpccm): x scale factor -> physical kpc
KEY = ["box", "snap", "gal_id"]


def read_anchor(box, z_target, snap):
    """One catalogue -> DataFrame of its galaxies with log M* > POOL_LOGM (gal_id = the galaxy's index in the catalogue)."""
    with h5py.File(cat_path(box, snap), "r") as f:
        sa = f["simulation_attributes"].attrs
        z, a = float(sa["redshift"]), float(sa["scale_factor"])
        g = f["galaxy_data"]
        ms = np.asarray(g["dicts/masses.stellar"][:], float)
        keep = np.where(ms > 10 ** POOL_LOGM)[0]
        d = {"gal_id": keep.astype(np.int64)}
        for col, cands in CAT_COLS.items():
            src = next((c for c in cands if c in g), None)
            d[col] = np.asarray(g[src][:], float)[keep] if src is not None else np.full(len(keep), np.nan)
    df = pd.DataFrame(d)
    for c in COMOVING_KPC:
        df[c] = df[c] * a
    df.insert(0, "box", box); df.insert(1, "z_target", float(z_target)); df.insert(2, "snap", int(snap)); df.insert(3, "z", z)
    df["t_H_gyr"] = float(COSMO.age(z).value)
    return df


def _load_cache():
    if OVERWRITE_CACHE or not os.path.exists(CACHE_FITS):
        return None
    t = Table.read(CACHE_FITS).to_pandas()
    t["box"] = [v.decode() if isinstance(v, bytes) else str(v) for v in t["box"]]
    t["box"] = t["box"].str.strip()
    return t


CAT = _load_cache()
_have = set(zip(CAT["box"], CAT["snap"].astype(int))) if CAT is not None else set()
_new = []
for box in ALL_BOXES:
    for zt, s in AVAIL[box]:
        if (box, int(s)) in _have:
            continue
        df = read_anchor(box, zt, s)
        _new.append(df)
        print(f"  read {box} snap {s:3d} (z = {df['z'].iloc[0]:.3f}): {len(df)} galaxies with log M* > {POOL_LOGM:g}")
if _new:
    CAT = pd.concat(([CAT] if CAT is not None else []) + _new, ignore_index=True)
    Table.from_pandas(CAT).write(CACHE_FITS, overwrite=True)
    print(f"cache -> {CACHE_FITS} ({len(CAT)} rows)")
elif CAT is not None:
    print(f"cache {CACHE_FITS}: {len(CAT)} rows, every available (box, anchor) present (OVERWRITE_CACHE = {OVERWRITE_CACHE})")
if CAT is None or not len(CAT):
    raise RuntimeError("no catalogue could be read: check SHARE / BOXES")
CAT = CAT.sort_values(KEY).reset_index(drop=True)


def m25_cuts(df):
    """The m25 Part 1 cuts on the catalogue columns at the anchor -> {name: mask}."""
    ms, sfr = df["mstar"].to_numpy(float), df["sfr"].to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ssfr = np.where(ms > 0, sfr / ms, np.nan)
        lm = np.log10(np.where(ms > 0, ms, np.nan))
    cuts = {"massive": lm > MASS_FLOOR,
            "passive": ssfr < PASSIVE_FACTOR / (df["t_H_gyr"].to_numpy(float) * 1e9),
            f"gas>={NGAS_MIN}": df["ngas"].to_numpy(float) >= NGAS_MIN,
            f"star>={NSTAR_MIN}": df["nstar"].to_numpy(float) >= NSTAR_MIN,
            "dust/H2": df["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * df["mh2"].to_numpy(float)}
    if CENTRALS_ONLY:
        cuts["central"] = df["central"].to_numpy(float) > 0
    return cuts


def derive(df):
    """The axes of the figure from the catalogue columns (one row per galaxy)."""
    G = df.copy()
    ms = G[MSTAR_COL].to_numpy(float)
    ms = np.where(ms > 0, ms, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        G["log_mstar"] = np.log10(G["mstar"].to_numpy(float))                     # the selection's M* (whole galaxy)
        G["fdust"] = G["mdust"].to_numpy(float) / ms                              # zero dust / H2 -> dropped on the log axes (never floored)
        G["fh2"] = HE * G["mh2"].to_numpy(float) / ms
        G["fgas"] = G["mgas"].to_numpy(float) / ms                               # whole-galaxy gas over stars (SIM_GAS_ROW)
        G["ssfr"] = np.maximum(G[SFR_COL].to_numpy(float), 0.0) / ms              # zero SFR kept: drawn at the floor
        G["re"] = R_PROJ_OVER_3D * G["r_half_star"].to_numpy(float)              # projected half-mass radius [kpc]
        G["sig_e"] = np.where(G["re"] > 0, 0.5 * ms / (np.pi * G["re"] ** 2), np.nan)
        mbh = G["mbh"].to_numpy(float)
        G["jet"] = (mbh > 0) & (np.log10(np.where(mbh > 0, mbh, 1.0)) > JET_LOGMBH) & (G["fedd"].to_numpy(float) < JET_FEDD)
        G["jet"] &= np.isfinite(G["fedd"].to_numpy(float))
    G["gkey"] = [f"{b}_{int(s)}_{int(g)}" for b, s, g in zip(G["box"], G["snap"], G["gal_id"])]
    return G


CUTS = m25_cuts(CAT)
_all = np.logical_and.reduce(list(CUTS.values()))
CAT["pop"] = np.where(_all, "Q", "")
Q = derive(CAT[_all])
QM = Q[Q["log_mstar"] > MASS_MIN].reset_index(drop=True)
_names = list(CUTS.keys())
print(f"\nfunnel per (box, anchor): the m25 cuts applied in sequence; Q = all of them; tracks = Q with log M* > {MASS_MIN:g}")
print(f"  {'box':10s} {'z':>5s} {'snap':>4s} " + " ".join(f"{n:>9s}" for n in _names) + f" {'Q':>5s} {'tracks':>6s} {'sat %':>6s}")
for box in ALL_BOXES:
    for zt, s in AVAIL[box]:
        m = (CAT["box"] == box).to_numpy() & (CAT["snap"] == int(s)).to_numpy()
        run, cells = m.copy(), []
        for n in _names:
            run &= CUTS[n]; cells.append(int(run.sum()))
        qm = m & _all
        sat = 100.0 * (1 - np.nanmean(CAT.loc[qm, "central"].to_numpy(float))) if qm.any() else np.nan
        print(f"  {box:10s} {zt:5.2f} {int(s):4d} " + " ".join(f"{c:9d}" for c in cells)
              + f" {int(qm.sum()):5d} {int((qm & (np.log10(np.where(CAT['mstar'] > 0, CAT['mstar'], np.nan)) > MASS_MIN)).sum()):6d} {sat:6.0f}")
print(f"\nthe tracks' sample (Q, log M* > {MASS_MIN:g}; whole-galaxy catalogue quantities, sSFR from `{SFR_COL}`, M* = `{MSTAR_COL}`, R_e = {R_PROJ_OVER_3D:g} x R_1/2(3-D)):")
for box in ALL_BOXES:
    g = QM[QM["box"] == box]
    if not len(g):
        print(f"  {box:10s} no galaxies"); continue
    print(f"  {box:10s} N = {len(g):4d} over {g['snap'].nunique()} anchors; satellites {100 * (1 - g['central'].mean()):.0f} %; zero dust {int((g['mdust'] <= 0).sum())}, "
          f"zero H2 {int((g['mh2'] <= 0).sum())}, zero SFR {int((g[SFR_COL] <= 0).sum())}; medians: log M* {g['log_mstar'].median():.2f}, age {g['age'].median():.2f} Gyr, "
          f"R_e {g['re'].median():.2f} kpc, log Sigma_e {np.log10(g['sig_e']).median():.2f}, log f_dust {np.log10(g.loc[g['fdust'] > 0, 'fdust']).median():.2f}, "
          f"log f_H2 {np.log10(g.loc[g['fh2'] > 0, 'fh2']).median():.2f}, log sSFR {np.log10(g['ssfr'].clip(lower=SSFR_FLOOR)).median():.2f}; "
          f"jet mode at the anchor {100 * g['jet'].mean():.0f} %")

## Part 2 — the figure: $M_{\rm dust}/M_\star$ and $M_{\rm H_2}/M_\star$ against age, $\Sigma_{\rm e}$ and sSFR, one track per box

The grid of figure H (rows = the expensive quantities, columns = the cheap ones; same axes, same limits). Per box the running median of the quenched galaxies under the mass cut in equal-count $x$ bins (`NBIN_X`, `NMIN_X`) with a bootstrap 16–84 % band; the Spearman $\rho$ over the galaxies is printed per panel and box. The m25 reference (`M25_TRACKS_CSV`, model `M25_REF_MODEL`: the whole-galaxy SIMBA truth of the m25 particle sample, figure H$_{\rm gal}$) is its *all quenched* track (black dotted; `SHOW_M25_CLASSES` adds its weak / intermediate / strong tracks as thin lines). The observed points are the ALMA-C11 sources (fiducial CIGALE, arrows = limits, controls small), the Spilker+18 LEGA-C passive galaxies (CO only) and ADF22-QG1, coloured by $A_V$; every detected value is compared with each box's track interpolated at its $x$ (`obs_vs_track` rows of the CSV, medians printed).

`paper_ism_prediction_boxes.{png,pdf}`, `paper_ism_prediction_boxes.csv` (model tracks, the m25 reference rows, `obs_vs_track`), `paper_ism_prediction_boxes_points.csv`.

In [ ]:
# ── Part 2 — the figure-H grid with one track per box, the m25 figure-H reference and the observed points ──
GRID = [[("age", "fdust"), ("sigma_e", "fdust"), ("ssfr", "fdust")], [("age", "fh2"), ("sigma_e", "fh2"), ("ssfr", "fh2")]]
SHOW_M25_ALLQ    = True       # the m25 "all quenched" reference track (M25_REF_MODEL: the whole-galaxy truth of the m25 particle sample, or figure H's CIGALE core fits)
SHOW_M25_CLASSES = False      # + its weak / intermediate / strong tracks (thin, class colours)
OBS_COLOUR, OBS_CLIM = "av", (0.0, 1.4)
OBS_CTRL  = True              # the six ALMA-C11 controls as small circles
RE_KIND   = "maj"             # observed R_e: "maj" semi-major (van der Wel+14 convention) | "circ" circularised
CMAP      = ("YlGnBu", 0.2, 1.0)
FONT      = dict(label=12.0, tick=10.0, legend=9.0, note=8.3, tag=10.5)
UMEHATA   = dict(name="ADF22-QG1", ref="Umehata+25", z=3.0922, logM=11.11, re_kpc=1.01, av=0.5, logage=8.79, logMH2=10.26, logMd_ul=8.02, ssfr_ul=1.8e-12)
# axes: col = plot-space column (log10 for log axes), raw = the linear column, floor = drawn there below (log axes; without one non-positive values are dropped)
AXES = {"av":      dict(col="lav", raw="av", log=True, lim=(2e-3, 4.0), label=r"$A_V$ [mag]", short="A_V"),
        "sigma_e": dict(col="lsig", raw="sig_e", log=True, lim=(1.5e8, 5e10), label=r"$\Sigma_{\rm e} = M_\star\,/\,2\pi R_{\rm e}^2$  [M$_\odot$ kpc$^{-2}$]", short="Sigma_e",
                        nm=r"$\Sigma_{\rm e}$", obs_note="semi-major $R_{\\rm e}$ (obs) / " + f"{R_PROJ_OVER_3D:g} x the 3-D stellar half-mass radius (cat.)"),
        "age":     dict(col="age", raw="age", log=False, lim=(0.3, 8.8), label="stellar age  [Gyr]", short="age",
                        obs_note="SED fit (obs) / mass-weighted, whole galaxy (cat.)"),
        "ssfr":    dict(col="lssfr", raw="ssfr", log=True, floor=SSFR_FLOOR, lim=(6e-14, 4e-10), cens="ssfr_censor", label=r"sSFR  [yr$^{-1}$]", short="sSFR",
                        obs_note=f"SED fit (obs) / `{SFR_COL}` over $M_\\star$ (cat.); < 1e-13 drawn at 1e-13"),
        "re":      dict(col="lre", raw="re_kpc", log=True, lim=(0.4, 40.0), label=r"$R_{\rm e}$  [kpc]", short="R_e"),
        "fdust":   dict(col="lfdust", raw="fdust", log=True, lim=(2e-6, 6e-3), cens="fdust_censor", label=r"$M_{\rm dust}\,/\,M_\star$", short="M_dust/M*", nm=r"$M_{\rm dust}/M_\star$",
                        obs_note="CIGALE $M_{\\rm dust}$ / $M_\\star$ (FAST++: ADF22-QG1) / whole-galaxy dust over stars (cat.)"),
        "fh2":     dict(col="lfh2", raw="fh2", log=True, lim=(2e-4, 6e-1), cens="fh2_censor", label=r"$M_{\rm H_2}\,/\,M_\star$", short="M_H2/M*", nm=r"$M_{\rm H_2}/M_\star$",
                        obs_note=r"$\alpha_{\rm CO}$ masses incl. He (obs) / 1.36 x the whole-galaxy H$_2$ over stars (cat.)")}
if SIM_GAS_ROW == "fgas":     # the H2 row carries the whole-galaxy gas fraction for the simulation and the alpha_CO H2 for the observations (`lfh2` stays the catalogue H2)
    AXES["fh2"].update(col="lgasrow", label=r"$M_{\rm H_2}/M_\star$ (obs)   $M_{\rm gas}/M_\star$ (sim)", short="M_gas/M* (sim; obs M_H2/M*)", nm=r"$M_{\rm gas}/M_\star$ (sim)",
                       obs_note=r"the whole-galaxy gas (every member gas particle, He incl.) over stars (cat.) against the $\alpha_{\rm CO}$ H$_2$ masses incl. He (obs);"
                                "\n" r"   not nested with the catalogue H$_2$ (summed over the halo gas assigned to the galaxy): median $M_{\rm gas}/1.36\,M_{\rm H_2}$ = +0.08 dex (cis100)")
SIM_GAS_COL = {"fh2": "fh2", "fgas": "fgas"}[SIM_GAS_ROW]
_cm0 = plt.get_cmap(CMAP[0])
CM = LinearSegmentedColormap.from_list(CMAP[0] + "_boxes", _cm0(np.linspace(CMAP[1], CMAP[2], 256)))
NORM = Normalize(*OBS_CLIM)
_cof = lambda v: CM(NORM(float(v))) if np.isfinite(v) else "white"


def _sigma_e(mstar, re_kpc):
    mstar, re_kpc = np.asarray(mstar, float), np.asarray(re_kpc, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isfinite(mstar) & np.isfinite(re_kpc) & (re_kpc > 0), 0.5 * mstar / (np.pi * re_kpc ** 2), np.nan)


def _fs(spec, v):
    """Raw values -> plot space (log10 on log axes; a floor if the axis has one, else non-positive values are dropped)."""
    v = np.asarray(v, float)
    if not spec["log"]:
        return v
    fl = spec.get("floor")
    with np.errstate(divide="ignore", invalid="ignore"):
        if fl:
            return np.where(np.isfinite(v), np.log10(np.maximum(v, fl)), np.nan)
        return np.where(np.isfinite(v) & (v > 0), np.log10(np.where(v > 0, v, 1.0)), np.nan)


def _raw(spec, v):
    return 10 ** np.asarray(v, float) if spec["log"] else np.asarray(v, float)


def _add_fit_cols(df, cols):
    for key, raw in cols.items():
        df[AXES[key]["col"]] = _fs(AXES[key], df[raw])
    return df


for _G in (Q, QM):
    _add_fit_cols(_G, {"sigma_e": "sig_e", "age": "age", "ssfr": "ssfr", "re": "re", "fdust": "fdust", "fh2": SIM_GAS_COL})
    _G["lfh2"] = _fs(AXES["fh2"], _G["fh2"])                                       # the catalogue H2 for the contrast tables, whatever the row shows

# ── the observed points (as figure H) ──
OBS = []
_gd = pd.read_csv(need(OBS_GD_CSV, "git pull")).set_index("id")
_st = pd.read_csv(need(OBS_STRUCT_CSV, "git pull")).set_index("id")
_tf = lambda v: str(v).strip().lower() in ("true", "1")
for sid, r in _gd.iterrows():
    if sid not in _st.index:
        print(f"  ALMA-C11 {sid}: no structure row -> skipped"); continue
    s = _st.loc[sid]
    re_ = float(s["re_kpc"]) if RE_KIND == "maj" else float(s["re_cosmosweb_kpc"] if np.isfinite(s["re_cosmosweb_kpc"]) else s["re_acs_kpc"])
    ctrl = str(sid).startswith("ctrl")
    if ctrl and not OBS_CTRL:
        continue
    M = 10 ** float(r["logMstar"])
    OBS.append(dict(sample="ALMA-C11 controls" if ctrl else "ALMA-C11", id=sid, z=float(r["z"]), av=float(r["AV_" + str(r["fid_run"]).strip()]), logM=float(r["logMstar"]),
                    re_kpc=re_, sig_e=float(_sigma_e(M, re_)), age=10 ** float(r["logage"]) / 1e9, ssfr=float(r["SFR"]) / M, ssfr_censor="det",
                    fdust=10 ** float(r["log_fdust"]), fdust_censor="ul" if _tf(r["dust_ul"]) else "det",
                    fh2=10 ** float(r["logMH2"]) / M, fh2_censor="ul" if _tf(r["co_ul"]) else "det",
                    marker="o", size=42 if ctrl else 120, edge="0.35" if ctrl else C_OBS_EDGE))
_lit = pd.read_csv(need(LIT_CSV, "git pull"))
_lit["re_use"] = _lit["re_5000_kpc"] if RE_KIND == "maj" else _lit["re_5000_kpc"] * np.sqrt(_lit["q"].clip(lower=0.05))
_lit["sig_e"] = _sigma_e(10 ** _lit["logM"].to_numpy(float), _lit["re_use"].to_numpy(float))
_lit["ssfr"] = np.where(_lit["logssfr"] > -50, 10 ** _lit["logssfr"].to_numpy(float), 0.0)
_lit["age"] = 10 ** _lit["logage_fit"].to_numpy(float)
_legac = _lit[_lit["sample"] == "LEGA-C"].drop_duplicates("id").set_index("id")
for _, r in pd.read_csv(need(SPILKER_CSV, "git pull")).iterrows():
    lid = int(str(r["galaxy"]).replace("LEGA-C", "").strip())
    if lid not in _legac.index:
        print(f"  Spilker+18 {r['galaxy']}: not in the LEGA-C literature rows -> skipped"); continue
    L = _legac.loc[lid]
    OBS.append(dict(sample="Spilker+18", id=f"LEGA-C {lid}", z=float(L["z"]), av=float(L["av"]), logM=float(L["logM"]), re_kpc=float(L["re_use"]), sig_e=float(L["sig_e"]),
                    age=float(L["age"]), ssfr=float(L["ssfr"]), ssfr_censor="det", fdust=np.nan, fdust_censor="none",
                    fh2=10 ** float(r["logMH2_h"]) / 10 ** float(L["logM"]), fh2_censor="det", marker="D", size=64, edge="0.2"))
U = UMEHATA
OBS.append(dict(sample=f"{U['name']} ({U['ref']})", id=U["name"], z=U["z"], av=U["av"], logM=U["logM"], re_kpc=U["re_kpc"], sig_e=float(_sigma_e(10 ** U["logM"], U["re_kpc"])),
                age=10 ** U["logage"] / 1e9, ssfr=U["ssfr_ul"], ssfr_censor="ul", fdust=10 ** (U["logMd_ul"] - U["logM"]), fdust_censor="ul",
                fh2=10 ** (U["logMH2"] - U["logM"]), fh2_censor="det", marker="*", size=300, edge="k"))
OBS = pd.DataFrame(OBS)
_add_fit_cols(OBS, {"av": "av", "sigma_e": "sig_e", "age": "age", "ssfr": "ssfr", "re": "re_kpc", "fdust": "fdust", "fh2": "fh2"})
OBS["lfh2"] = _fs(AXES["fh2"], OBS["fh2"])                                          # the observed H2 under its own name too (Part 4 dust-to-H2)
print(f"observed points: {int((OBS['sample'] == 'ALMA-C11').sum())} ALMA-C11 QGs + {int((OBS['sample'] == 'ALMA-C11 controls').sum())} controls, "
      f"{int((OBS['sample'] == 'Spilker+18').sum())} Spilker+18, {U['name']}")

# ── the m25 reference tracks: M25 in the CSV schema of the paper notebook (kind / panel / sample / x / y / y_lo16 / y_hi84 / n_gal_group, linear values) — filled below ──
M25, M25_SAMPLE = None, None
M25_REF = ([("SIMBA all quenched", dict(color="0.1", ls=(0, (1.2, 1.6)), lw=2.2, label=f"{M25_REF_LABEL}: all quenched"))] if SHOW_M25_ALLQ else []) + \
          ([(f"SIMBA {c}", dict(color=CLASS_COLOR[c], ls="-", lw=1.1, label=f"{M25_REF_LABEL}: {c} AGN coupling")) for c in CLASSES] if SHOW_M25_CLASSES else [])


def m25_track(panel, sample):
    if M25 is None:
        return pd.DataFrame(columns=["x", "y", "lo", "hi", "n"])
    t = M25[M25["kind"].isin(["model_track", "model_track_allQ"]) & (M25["panel"] == panel) & (M25["sample"] == sample)]
    return pd.DataFrame({"x": t["x"].to_numpy(float), "y": t["y"].to_numpy(float), "lo": t["y_lo16"].to_numpy(float), "hi": t["y_hi84"].to_numpy(float),
                         "n": t["n_gal_group"].to_numpy(float)})


def run_track(x, y, nbins=NBIN_X, nmin=NMIN_X, nmin_small=5, n=NBOOT, seed=0):
    """Running median of y in equal-count x bins (plot-space columns, one row per galaxy) with a bootstrap 16–84 % band -> (x, y, lo, hi, n);
    a sample smaller than 2 x nmin falls back to 4 bins of >= min(nmin, nmin_small)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 2 * nmin:
        nbins, nmin = 4, min(nmin, nmin_small)
    nb = min(nbins, len(x) // nmin)
    if nb < 2:
        return pd.DataFrame(columns=["x", "y", "lo", "hi", "n"])
    e = np.quantile(x, np.linspace(0, 1, nb + 1))
    ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb - 1)
    rng, out = np.random.default_rng(seed), []
    for b in range(nb):
        yy = y[ib == b]
        bs = np.median(yy[rng.integers(0, len(yy), (n, len(yy)))], axis=1)
        out.append((np.median(x[ib == b]), np.median(yy), np.percentile(bs, 16), np.percentile(bs, 84), len(yy)))
    return pd.DataFrame(out, columns=["x", "y", "lo", "hi", "n"])


def _draw_track(ax, tr, xs, ys, st, band=True, z=4):
    X, Y = _raw(xs, tr["x"]), _raw(ys, tr["y"])
    if band:
        ax.fill_between(X, _raw(ys, tr["lo"]), _raw(ys, tr["hi"]), color=st["color"], alpha=0.14, lw=0, zorder=z - 1)
    ax.plot(X, Y, color=st["color"], lw=st.get("lw", 2.8), ls=st.get("ls", "-"), marker=st.get("marker", None), ms=7, mec="k", mew=0.6, zorder=z,
            path_effects=[withStroke(linewidth=st.get("lw", 2.8) + 1.8, foreground="white")])


def _rows_of(rows, kind, pan, sample, tr, ng, xs, ys):
    for _, r in tr.iterrows():
        rows.append(dict(kind=kind, panel=pan, sample=sample, n_gal_group=ng, x=float(_raw(xs, r["x"])), y=float(_raw(ys, r["y"])),
                         y_lo16=float(_raw(ys, r["lo"])), y_hi84=float(_raw(ys, r["hi"])), n_gal=int(r["n"])))


def draw_obs(ax, xs, ys):
    """The observed points on one plane (colour = A_V, arrows = limits)."""
    for _, r in OBS.iterrows():
        if not (np.isfinite(r[xs["col"]]) and np.isfinite(r[ys["col"]])):
            continue
        x, y = float(_raw(xs, r[xs["col"]])), float(_raw(ys, r[ys["col"]]))
        ax.scatter(x, y, s=r["size"], marker=r["marker"], c=[_cof(r[AXES[OBS_COLOUR]["raw"]])], edgecolors=r["edge"], linewidths=1.4, zorder=6 + (r["marker"] == "*"))
        if ys.get("cens") and r[ys["cens"]] == "ul":
            ax.annotate("", (x, y), xytext=(0, -15), textcoords="offset points", arrowprops=dict(arrowstyle="<|-", color=r["edge"], lw=1.2, mutation_scale=9), zorder=8)
        if xs.get("cens") and r[xs["cens"]] == "ul":
            ax.annotate("", (x, y), xytext=(-15, 0), textcoords="offset points", arrowprops=dict(arrowstyle="<|-", color=r["edge"], lw=1.2, mutation_scale=9), zorder=8)


def ism_panel(ax, xkey, ykey, groups, rows, tag, xlab=True, ylab=True, nbins=NBIN_X, nmin=NMIN_X):
    """One plane: the model tracks of `groups` (name, frame, style), the m25 reference tracks, the observed points -> {name: n_gal} of the tracks drawn."""
    xs, ys = AXES[xkey], AXES[ykey]
    pan = f"{xkey}_{ykey}"
    drawn, tracks = {}, {}
    ax.set_xscale("log" if xs["log"] else "linear"); ax.set_yscale("log" if ys["log"] else "linear")
    ax.set_xlim(*xs["lim"]); ax.set_ylim(*ys["lim"])
    for sample, st in M25_REF:
        tr = m25_track(pan, sample)
        if len(tr):
            trp = pd.DataFrame({"x": _fs(xs, tr["x"]), "y": _fs(ys, tr["y"]), "lo": _fs(ys, tr["lo"]), "hi": _fs(ys, tr["hi"]), "n": tr["n"]})
            _draw_track(ax, trp, xs, ys, st, band=False, z=3)
            _rows_of(rows, "m25_reference", pan, sample, trp, int(tr["n"].iloc[0]), xs, ys)
    for name, frame, st in groups:
        tr = run_track(frame[xs["col"]], frame[ys["col"]], nbins=nbins, nmin=nmin)
        ok = np.isfinite(frame[xs["col"]]) & np.isfinite(frame[ys["col"]])
        rho = spearmanr(frame.loc[ok, xs["col"]], frame.loc[ok, ys["col"]]) if ok.sum() >= 5 else (np.nan, np.nan)
        print(f"  {pan:14s} {name:24s} {int(ok.sum()):4d} galaxies, {len(tr)} track points" + (f"; {ys['short']} {_raw(ys, tr['y'].min()):.3g}–{_raw(ys, tr['y'].max()):.3g} over "
              f"{xs['short']} {_raw(xs, tr['x'].min()):.3g}–{_raw(xs, tr['x'].max()):.3g}" if len(tr) else "") + f"; Spearman = {rho[0]:+.2f} (p = {rho[1]:.2g})")
        if len(tr):
            drawn[name], tracks[name] = int(ok.sum()), tr
            _draw_track(ax, tr, xs, ys, st, band=st.get("band", True))
            _rows_of(rows, "model_track", pan, name, tr, int(ok.sum()), xs, ys)
    draw_obs(ax, xs, ys)
    if ys.get("cens"):    # the check: detected values against each track interpolated at their x (inside the track's range)
        for name, tr in tracks.items():
            O = OBS[(OBS[ys["cens"]] == "det") & np.isfinite(OBS[xs["col"]]) & np.isfinite(OBS[ys["col"]]) & OBS[xs["col"]].between(tr["x"].min(), tr["x"].max())]
            if len(O) >= 3:
                d = O[ys["col"]].to_numpy(float) - np.interp(O[xs["col"]].to_numpy(float), tr["x"].to_numpy(float), tr["y"].to_numpy(float))
                print(f"    measured − {name} track over the {len(O)} detections inside its {xs['short']} range: median {np.median(d):+.2f} dex, 16–84 % [{np.percentile(d, 16):+.2f}, {np.percentile(d, 84):+.2f}]")
                for _, o in O.iterrows():
                    yt = float(np.interp(o[xs["col"]], tr["x"], tr["y"]))
                    rows.append(dict(kind="obs_vs_track", panel=pan, sample=o["sample"], id=o["id"], track=name, x=float(_raw(xs, o[xs["col"]])), y=float(_raw(ys, o[ys["col"]])),
                                     y_track=float(_raw(ys, yt)), offset_dex=float(o[ys["col"]] - yt)))
    ax.set_xlabel(xs["label"] if xlab else "", fontsize=FONT["label"]); ax.set_ylabel(ys["label"] if ylab else "", fontsize=FONT["label"])
    ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=FONT["tick"], labelbottom=xlab, labelleft=ylab); ax.grid(False)
    for axis, spec in ((ax.yaxis, ys), (ax.xaxis, xs)):
        if spec["log"] and spec["lim"][0] >= 0.1 and spec["lim"][1] < 1e4:
            axis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:g}"))
    if tag:
        ax.set_title(tag, loc="left", fontsize=FONT["tag"], pad=6)
    return drawn


def ism_figure(stem, groups, model_title, extra_notes=(), nbins=NBIN_X, nmin=NMIN_X):
    """The 2 x 3 grid + the legend strip (observed | models | definitions) + the A_V colour bar -> <stem>.{png,pdf,csv,_points.csv};
    returns (what was drawn per panel, the CSV rows: model_track / m25_reference / obs_vs_track)."""
    print(f"\n── {stem}: " + " | ".join(", ".join(f"{AXES[y]['short']} vs {AXES[x]['short']}" for x, y in row) for row in GRID) + " ──")
    rows, nr, nc = [], len(GRID), max(len(r) for r in GRID)
    fig = plt.figure(figsize=(4.9 * nc + 1.6, 4.1 * nr + 2.9))
    gs = GridSpec(nr + 1, nc, height_ratios=[1.0] * nr + [0.46], hspace=0.10, wspace=0.08, figure=fig)
    axes = [[fig.add_subplot(gs[i, j]) for j in range(len(GRID[i]))] for i in range(nr)]
    lax, lax2, lax3 = fig.add_subplot(gs[nr, 0]), fig.add_subplot(gs[nr, 1]), fig.add_subplot(gs[nr, 2:])
    for a in (lax, lax2, lax3):
        a.axis("off")
    drawn, k = [], 0
    for i, row in enumerate(GRID):
        for j, (x, y) in enumerate(row):
            drawn.append(ism_panel(axes[i][j], x, y, groups, rows, f"({'abcdefghij'[k]})", xlab=(i == nr - 1), ylab=(j == 0), nbins=nbins, nmin=nmin)); k += 1
    flat = [a for row in axes for a in row]
    sm = plt.cm.ScalarMappable(cmap=CM, norm=NORM); sm.set_array([])
    cb = fig.colorbar(sm, ax=flat, pad=0.012, extend="max", fraction=0.02, aspect=40)
    cb.set_label(AXES[OBS_COLOUR]["label"] + " of the observed galaxies", fontsize=FONT["label"]); cb.ax.tick_params(labelsize=FONT["tick"])
    nq, ns = int((OBS["sample"] == "ALMA-C11").sum()), int((OBS["sample"] == "Spilker+18").sum())
    h_obs = [(Line2D([], [], marker="o", ls="", ms=10, mfc=_cof(np.mean(OBS_CLIM)), mec=C_OBS_EDGE, mew=1.3), rf"ALMA-C11 QGs, $z\approx0.4$ (N={nq}): colour = $A_V$ (fiducial CIGALE)" + "\narrow = upper limit (dust, CO, sSFR)"),
             (Line2D([], [], marker="D", ls="", ms=7.5, mfc=_cof(0.3), mec="0.2", mew=1.3), rf"Spilker+18 LEGA-C passive, $z\approx0.7$ (N={ns}): CO only"),
             (Line2D([], [], marker="*", ls="", ms=15, mfc=_cof(U["av"]), mec="k", mew=1.2), rf"{U['name']} ({U['ref']}), $z = {U['z']:.2f}$: $A_V$ = {U['av']:g} (FAST++)")]
    if OBS_CTRL:
        h_obs.insert(1, (Line2D([], [], marker="o", ls="", ms=6.5, mfc=_cof(0.15), mec="0.35", mew=1.2), "ALMA-C11 controls (no ALMA detection: limits only)"))
    ng = {name: max(d.get(name, 0) for d in drawn) for name, _, _ in groups}
    h_mod = [(Line2D([], [], color=st["color"], lw=st.get("lw", 2.8), ls=st.get("ls", "-"), marker=st.get("marker", None), ms=7, mec="k", mew=0.6), f"{st['label']} (N={ng[name]})")
             for name, _, st in groups if ng[name]]
    h_mod += [(Line2D([], [], color=st["color"], lw=st["lw"], ls=st["ls"]), st["label"] + (f" (N={int(m25_track('age_fdust', sample)['n'].iloc[0])})" if len(m25_track("age_fdust", sample)) else ""))
              for sample, st in M25_REF if M25 is not None and len(m25_track("age_fdust", sample))]
    leg1 = lax.legend([h for h, _ in h_obs], [l for _, l in h_obs], loc="upper left", bbox_to_anchor=(-0.04, 0.80), frameon=False, fontsize=FONT["legend"], handlelength=1.6,
                      title="observed", title_fontsize=FONT["legend"] + 1)
    leg1._legend_box.align = "left"
    leg2 = lax2.legend([h for h, _ in h_mod], [l for _, l in h_mod], loc="upper left", bbox_to_anchor=(-0.02, 0.80), frameon=False, fontsize=FONT["legend"], handlelength=2.2,
                       title=model_title, title_fontsize=FONT["legend"] + 0.5)
    leg2._legend_box.align = "left"
    keys_used = list(dict.fromkeys([x for row in GRID for x, y in row] + [y for row in GRID for x, y in row]))
    notes = [f"{AXES[kk].get('nm', AXES[kk]['short'])}: {AXES[kk]['obs_note']}" for kk in keys_used if AXES[kk].get("obs_note")]
    notes.append(f"catalogues: whole-galaxy caesar quantities, one row per galaxy,\n   <= {nbins} equal-count bins of >= {nmin} galaxies (fewer than {2 * nmin}: 4 bins of >= {min(nmin, 5)})")
    notes += list(extra_notes)
    lax3.text(0.02, 0.80, "\n".join(["definitions"] + notes), transform=lax3.transAxes, fontsize=FONT["note"], color="0.35", ha="left", va="top", linespacing=1.45)
    paper_save(fig, stem)
    plt.show()
    pd.DataFrame(rows).to_csv(os.path.join(OUT, f"{stem}.csv"), index=False)
    OBS.to_csv(os.path.join(OUT, f"{stem}_points.csv"), index=False)
    print(f"  tables -> {os.path.join(OUT, stem)}{{.csv,_points.csv}}")
    return drawn, pd.DataFrame(rows)


def m25_catalogue_reference():
    """The m25 particle sample as the catalogue sees it: the quenched galaxies of the m25 selection table (pop Q, the selection's log M* > MASS_MIN) matched to the cis25
    rows of Q, their class from the selection's w_pre under CLASS_SCHEME -> (tracks in the paper CSV schema: model_track_allQ = all quenched, model_track = per class; the rows)."""
    if not os.path.exists(M25_SELECTION):
        print(f"no {os.path.relpath(M25_SELECTION, os.getcwd())}: the m25 catalogue reference is not drawn (m25 Part 3 selection table)"); return None, None
    t = Table.read(M25_SELECTION); t = t[np.char.strip(np.asarray(t["pop"]).astype(str)) == "Q"]
    sel = pd.DataFrame({"snap": np.asarray(t["snap"]).astype(int), "gal_id": np.asarray(t["gal_id"]).astype(int), "w_pre_sel": np.asarray(t["w_pre"], float),
                        "class3_sel": np.char.strip(np.asarray(t["agn_class_pt"]).astype(str)), "log_mstar_sel": np.asarray(t["log_mstar"], float)})
    sel = sel[sel["log_mstar_sel"] > MASS_MIN]
    Sm = Q[Q["box"] == "cis25"].merge(sel, on=["snap", "gal_id"], how="inner").copy()
    Sm["cls"] = [coupling_class(w) if np.isfinite(w) and c3 in CLASSES3 else c3 for w, c3 in zip(Sm["w_pre_sel"], Sm["class3_sel"])]
    print(f"m25 reference (catalogue): {len(Sm)} of the {len(sel)} quenched galaxies of the m25 sample with log M* > {MASS_MIN:g} found among the cis25 catalogue rows of Q"
          f" (the others fail the catalogue rule or sit at anchors not read); classes ({CLASS_SCHEME}): " + ", ".join(f"{c} {int((Sm['cls'] == c).sum())}" for c in CLASSES)
          + f", other {int((~Sm['cls'].isin(CLASSES)).sum())}")
    rows = []
    for row in GRID:
        for xk, yk in row:
            xs, ys = AXES[xk], AXES[yk]
            for kind, sample, frame in [("model_track_allQ", "SIMBA all quenched", Sm)] + [("model_track", f"SIMBA {c}", Sm[Sm["cls"] == c]) for c in CLASSES]:
                tr = run_track(frame[xs["col"]], frame[ys["col"]], nbins=M25_REF_NBIN, nmin=M25_REF_NMIN)
                _rows_of(rows, kind, f"{xk}_{yk}", sample, tr, int((np.isfinite(frame[xs["col"]]) & np.isfinite(frame[ys["col"]])).sum()), xs, ys)
    return pd.DataFrame(rows), Sm


if M25_REF_MODEL == "catalogue":
    M25, M25_SAMPLE = m25_catalogue_reference()
elif os.path.exists(M25_TRACKS_CSV):
    M25 = pd.read_csv(M25_TRACKS_CSV)
    print(f"m25 reference ({M25_REF_MODEL}): {os.path.relpath(M25_TRACKS_CSV, os.getcwd())} ({int(M25['kind'].isin(['model_track', 'model_track_allQ']).sum())} track points; "
          f"samples {sorted(set(M25.loc[M25['kind'].isin(['model_track', 'model_track_allQ']), 'sample']))}; its classes are the rule's three)")
else:
    print(f"no {os.path.relpath(M25_TRACKS_CSV, os.getcwd())}: the m25 reference tracks ({M25_REF_MODEL}) are not drawn (run paper_figures_quenched_m25 Part 6)")

BOX_GROUPS = [(box, QM[QM["box"] == box], dict(BOXES[box], lw=2.8)) for box in BOX_ORDER if (QM["box"] == box).sum() >= 10]
SEL_TXT = (f"log $M_\\star$ > {MASS_MIN:g}, passive (sSFR < {PASSIVE_FACTOR:g} / $t_H$),\n>= {NGAS_MIN} gas and >= {NSTAR_MIN} star particles, "
           f"$M_{{\\rm dust}}$ >= {DUST_TO_H2_MIN:g} $M_{{\\rm H_2}}$")
DRAWN_BOXES, ROWS_BOXES = ism_figure("paper_ism_prediction_boxes", BOX_GROUPS,
                                     f"SIMBA quenched galaxies from the catalogues, z = {min(ANCHORS):g}–{max(ANCHORS):g}:\n{SEL_TXT}\nrunning median per box (band = bootstrap 16–84 %)",
                                     extra_notes=[M25_REF_NOTE + f"; its own selection, {M25_REF_NBIN} bins"])

## Part 3 — the only AGN state a catalogue carries: the jet criterion at the anchor

The m25 classes (weak / intermediate / strong coupling) are set on the pre-quenching history and cannot be rebuilt from a catalogue. What a catalogue does carry is the black hole at the anchor: `masses.bh` and `bh_fedd`, hence SIMBA's own jet criterion ($\log M_{\rm BH} > 7.5$ and $f_{\rm Edd} < 0.2$). The same grid for `P3_BOX`, the quenched galaxies split by that state, with the m25 reference class tracks (whole-galaxy truth, `M25_REF_MODEL`) as thin lines; the Mann–Whitney contrasts of the two groups (mass, age, $\Sigma_{\rm e}$, sSFR, dust and H$_2$ fractions) are printed, and the jet fraction per box and anchor. `paper_ism_prediction_agnstate_<box>.{png,pdf,csv}`.

In [ ]:
# ── Part 3 — the same grid for one box, the quenched galaxies split by the AGN state at the anchor (the catalogue's only AGN information) ──
P3_BOX = "cis100"
P3_SPLIT = "three"            # "two": jet mode vs the rest | "three": the rest split into its two physical states (BH below the mass threshold / f_Edd >= 0.2)
P3_STYLE = {"jet":    dict(color="#7b3294", ls="-", marker="o", lw=2.8, label=rf"jet mode at the anchor ($\log M_{{\rm BH}} > {JET_LOGMBH:g}$, $f_{{\rm Edd}} < {JET_FEDD:g}$)"),
            "no_jet": dict(color="#008837", ls=(0, (4.0, 1.6)), marker="s", lw=2.8, label="not in jet mode"),
            "low_bh": dict(color="#008837", ls=(0, (4.0, 1.6)), marker="s", lw=2.8, label=rf"no jet: $\log M_{{\rm BH}} \leq {JET_LOGMBH:g}$ (BH not grown; incl. no BH)"),
            "high_fedd": dict(color="#e6ab02", ls=(0, (1.5, 1.2)), marker="^", lw=2.8, label=rf"no jet: $\log M_{{\rm BH}} > {JET_LOGMBH:g}$ and $f_{{\rm Edd}} \geq {JET_FEDD:g}$ (radiative mode)")}
P3_REF_CLASSES = ["weak", "strong"]   # figure-H class tracks drawn as the reference (thin lines)

print(f"jet-mode fraction of the tracks' sample per anchor (log M_BH > {JET_LOGMBH:g}, f_Edd < {JET_FEDD:g}):")
print(f"  {'box':10s} " + " ".join(f"{'z=' + str(zt):>7s}" for zt in ANCHORS) + f" {'all':>7s}")
for box in ALL_BOXES:
    g = QM[QM["box"] == box]
    cells = [(f"{100 * g.loc[g['snap'] == s, 'jet'].mean():5.0f} %" if (g["snap"] == s).any() else f"{'—':>7s}") for zt, s in ANCHORS.items()]
    print(f"  {box:10s} " + " ".join(f"{c:>7s}" for c in cells) + (f" {100 * g['jet'].mean():5.0f} %" if len(g) else f" {'—':>7s}"))


def agn_state(g):
    """'jet' | 'low_bh' (M_BH at or below the jet mass threshold, or no BH) | 'high_fedd' (massive BH accreting at f_Edd >= JET_FEDD) per row."""
    mbh = g["mbh"].to_numpy(float)
    big = (mbh > 0) & (np.log10(np.where(mbh > 0, mbh, 1.0)) > JET_LOGMBH)
    return np.where(g["jet"].to_numpy(bool), "jet", np.where(big, "high_fedd", "low_bh"))


QM["agn_state"] = agn_state(QM)
_VARS = (("log_mstar", "log M*"), ("age", "age [Gyr]"), ("lsig", "log Sigma_e"), ("lssfr", "log sSFR"), ("lfdust", "log M_dust/M*"), ("lfh2", "log M_H2/M*"),
         ("kappa_gas", "kappa_rot gas"), ("kappa_star", "kappa_rot stars"), ("z", "z"), ("central", "central (1) / sat (0)"))


def _stat(col):
    return (lambda v: v.mean()) if col == "central" else (lambda v: v.median())   # the central flag: its mean = the central fraction


def agn_state_contrasts(box, g3):
    """Split `g3` (rows of one box, `agn_state` set) into jet / low_bh / high_fedd (+ no_jet), print the medians per state with the Mann–Whitney p against the jet group -> {state: frame}."""
    grp = {k: g3[g3["agn_state"] == k] for k in ("jet", "low_bh", "high_fedd")}
    grp["no_jet"] = g3[g3["agn_state"] != "jet"]
    print(f"\n{box}: " + ", ".join(f"{k} {len(v)}" for k, v in grp.items()) + " quenched galaxies under the mass cut (medians; Mann–Whitney p against the jet-mode group):")
    print(f"  {'':>16s} " + " ".join(f"{k:>22s}" for k in ("jet", "low_bh", "high_fedd")))
    for col, lab in _VARS:
        a = grp["jet"][col].dropna()
        cells = [f"{_stat(col)(a):7.3f} (n={len(a):4d})      " if len(a) else f"{'—':>22s}"]
        for k in ("low_bh", "high_fedd"):
            b = grp[k][col].dropna()
            p = mannwhitneyu(a, b).pvalue if min(len(a), len(b)) >= 2 else np.nan
            cells.append(f"{_stat(col)(b):7.3f} (n={len(b):3d}) p={p:.1g}" if len(b) else f"{'—':>22s}")
        print(f"  {lab:>16s} " + " ".join(f"{c:>22s}" for c in cells))
    return grp


_grp = agn_state_contrasts(P3_BOX, QM[QM["box"] == P3_BOX].copy())
print("  per box, the dust fraction of the three states (medians of log M_dust/M*):")
for box in ALL_BOXES:
    g = QM[QM["box"] == box]
    print(f"    {box:10s} " + "  ".join(f"{k} {g.loc[g['agn_state'] == k, 'lfdust'].median():6.2f} (n={int((g['agn_state'] == k).sum()):4d})" for k in ("jet", "low_bh", "high_fedd")))

_ref_save = M25_REF
M25_REF = ([("SIMBA all quenched", dict(color="0.1", ls=(0, (1.2, 1.6)), lw=2.2, label=f"{M25_REF_LABEL}: all quenched"))] if SHOW_M25_ALLQ else []) + \
          [(f"SIMBA {c}", dict(color=CLASS_COLOR[c], ls="-", lw=1.1, label=f"{M25_REF_LABEL}: {c} AGN coupling")) for c in P3_REF_CLASSES]
_keys = ["jet", "no_jet"] if P3_SPLIT == "two" else ["jet", "low_bh", "high_fedd"]
P3_GROUPS = [(k, _grp[k], P3_STYLE[k]) for k in _keys if len(_grp[k]) >= 10]
DRAWN_P3, ROWS_P3 = ism_figure(f"paper_ism_prediction_agnstate_{P3_BOX}", P3_GROUPS,
                      f"SIMBA {BOXES[P3_BOX]['label']} quenched galaxies (catalogue),\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nsplit by the AGN state at the anchor",
                      extra_notes=["AGN state = SIMBA's jet criterion on the catalogue M_BH and f_Edd at the anchor\n   (no history: not the m25 coupling classes)",
                                   M25_REF_NOTE])
M25_REF = _ref_save

## Part 4 — the 50 Mpc feedback variants: which physics puts the quenched galaxies where the data are

The same box, resolution and selection with one AGN channel removed at a time (`VARIANTS`; the catalogues of `SIMBA_50/<variant>/Groups`, ROE `m50n512/<variant>/catalogs`): **s50** = the fiducial physics (AGN winds + jets + X-ray heating), **s50nox** = no X-ray feedback, **s50noagn** = no AGN feedback at all (`cis50nojet`, jets and X-ray off, is configured — add it to `VARIANTS`). One track per variant on the figure-H grid with finer bins (`P4_NBIN`, `P4_NMIN`: 8 equal-count bins of $\geq 3$ galaxies), the m25 all-quenched reference track (whole-galaxy truth), the observed points as before. The variant without a channel has *fewer* quenched galaxies (the funnel of Part 1 shows the passive fraction collapse without jets), so what it shows is the ISM of the galaxies that still quench without that channel.

The check is the table printed at the end: per panel and variant, the median offset of the detected observed values from the track interpolated at their $x$ (`obs_vs_track` rows, inside the track's range) and, per variant, the median $|{\rm offset}|$ over the dust and the H$_2$ panels — the variant closest to zero is the one whose quenched population looks most like the observed dusty / gas-rich QGs. `paper_ism_prediction_variants_m50.{png,pdf,csv,_points.csv}`.

In [ ]:
# ── Part 4 — the 50 Mpc feedback variants on the figure-H grid: one track per variant, finer bins, the offsets of the detections per variant ──
P4_NBIN, P4_NMIN = 8, 3        # tracks: up to 8 equal-count x bins of >= 3 galaxies (the variants' quenched samples are small)
P4_MIN_GAL = 6                 # a variant with fewer quenched galaxies under the cuts is listed but not drawn

P4_GROUPS = []
print(f"the 50 Mpc variants under the cuts (Q, log M* > {MASS_MIN:g}):")
for v in VARIANTS:
    g = QM[QM["box"] == v]
    if not len(g):
        print(f"  {v:10s} no catalogue read / no quenched galaxy (anchors available: {len(AVAIL.get(v, []))})"); continue
    print(f"  {v:10s} N = {len(g):4d} over {g['snap'].nunique():2d} anchors (z {', '.join(f'{z:g}' for z in sorted(set(g['z_target'])))}); satellites {100 * (1 - g['central'].mean()):.0f} %; "
          f"jet mode {100 * g['jet'].mean():.0f} %; medians: log M* {g['log_mstar'].median():.2f}, age {g['age'].median():.2f} Gyr, log Sigma_e {g['lsig'].median():.2f}, "
          f"log sSFR {g['lssfr'].median():.2f}, log f_dust {g['lfdust'].median():.2f}, log f_H2 {g['lfh2'].median():.2f}, kappa_gas {g['kappa_gas'].median():.2f}")
    if len(g) >= P4_MIN_GAL:
        P4_GROUPS.append((v, g, dict(BOXES[v], lw=2.8)))

if len(P4_GROUPS) < 1:
    print("nothing to draw: download the variant catalogues into SHARE/SIMBA_50/<variant>/Groups (ROE_CATALOG_URL) and re-run Part 1")
else:
    DRAWN_P4, ROWS_P4 = ism_figure("paper_ism_prediction_variants_m50", P4_GROUPS,
                                   f"SIMBA m50n512 feedback variants (catalogues), quenched:\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nrunning median per variant (band = bootstrap 16–84 %)",
                                   extra_notes=["variants: the same box, resolution and selection with one AGN channel removed\n   (the variant's own quenched population, not the fiducial galaxies re-simulated)",
                                                M25_REF_NOTE],
                                   nbins=P4_NBIN, nmin=P4_NMIN)
    # the check: median offset of the detected observed values from each variant's track, per panel; then per variant over the dust / H2 panels
    O = ROWS_P4[ROWS_P4["kind"] == "obs_vs_track"]
    names = [n for n, _, _ in P4_GROUPS]
    panels = [f"{x}_{y}" for row in GRID for x, y in row]
    print(f"\nmeasured − track [dex] over the detections inside each track's x range (median, N): the variant closest to 0 puts its quenched galaxies where the data are")
    print(f"  {'panel':14s} " + " ".join(f"{n:>18s}" for n in names))
    for pan in panels:
        cells = []
        for n in names:
            d = O[(O["panel"] == pan) & (O["track"] == n)]["offset_dex"]
            cells.append(f"{d.median():+6.2f} (N={len(d):2d})" if len(d) else f"{'—':>13s}")
        print(f"  {pan:14s} " + " ".join(f"{c:>18s}" for c in cells))
    for lab, ys in (("dust panels", "fdust"), ("H2 panels", "fh2")):
        cells = []
        for n in names:
            d = O[O["panel"].str.endswith("_" + ys) & (O["track"] == n)]["offset_dex"]
            cells.append(f"{d.abs().median():6.2f} (N={len(d):2d})" if len(d) else f"{'—':>13s}")
        print(f"  {'median |offset| ' + lab:>32s} " + " ".join(f"{c:>18s}" for c in cells))

## Part 5 — the AGN states inside each 50 Mpc variant: is there one population with the dust of the no-feedback runs *and* the H$_2$ of the fiducial run?

Part 4 leaves a contradiction: the runs without X-ray / AGN feedback put their quenched galaxies on the observed dust fractions (offsets $\sim 0.1$ dex) but $0.4$–$0.5$ dex *below* the observed H$_2$ fractions, while the fiducial run reproduces the H$_2$ and sits $1.5$–$1.9$ dex below the dust. Part 3 hinted at a way out in the 100 Mpc box: the quenched galaxies whose BH has *not* grown past the jet threshold (`low_bh`) keep $30\times$ more dust than the jet-mode ones and are within $\sim 0.25$ dex of the detections in both rows. This part runs the same split (`agn_state`: jet / `low_bh` / `high_fedd` at the anchor) inside each 50 Mpc variant (`VARIANTS`) with the Part 4 binning:

* **per variant** — the Part 3 contrasts (medians per state, Mann–Whitney against the jet group) and the figure-H grid split by state $\to$ `paper_ism_prediction_agnstate_m50_<variant>.*`;
* **state $\times$ variant** — one figure with the jet and `low_bh` tracks of every variant (colour = state, line style / marker = variant) $\to$ `paper_ism_prediction_agnstate_variants_m50.*`; since the state is the *same* $M_{\rm BH}$ / $f_{\rm Edd}$ criterion in every run, comparing a state across variants isolates what the missing channel does to galaxies in that state (in `s50nox` the jets act without the X-ray heating; in `s50noagn` nothing acts — its "jet mode" is a BH-growth label only);
* **the scorecard** — every population (variant, all / per state) with the median offset of the detections from its track over the dust panels and over the H$_2$ panels, its median $\log M_{\rm dust}/M_{\rm H_2}$ against the observed one, ranked by the larger of the two $|{\rm offsets}|$: the population closest to zero in *both* rows is the one that looks like the observed dusty, gas-rich QGs. Populations with fewer than `P5_MIN_GAL` galaxies are listed but not drawn.

In [ ]:
# ── Part 5 — the AGN states inside each 50 Mpc variant: per-variant split, state × variant figure, the joint dust + H2 scorecard ──
P5_NBIN, P5_NMIN = P4_NBIN, P4_NMIN    # the Part 4 binning (up to 8 equal-count bins of >= 3 galaxies)
P5_MIN_GAL = 6                          # a (variant, state) population with fewer galaxies is listed, not drawn
P5_BAND_MIN = 30                        # the bootstrap band is drawn only for populations of at least this many galaxies (the small ones' bands fill the panel)
P5_STATES = ["jet", "low_bh", "high_fedd"]
P5_CROSS_STATES = ["jet", "low_bh"]     # the states compared across variants in the combined figure (high_fedd has <= 3 galaxies per variant)
P5_SHORT = {"cis50": "s50", "cis50nox": "s50nox", "cis50noagn": "s50noagn", "cis50nojet": "s50nojet"}
P5_VARIANT_LS = {"cis50": "-", "cis50nox": (0, (4.0, 1.6)), "cis50noagn": (0, (1.5, 1.2)), "cis50nojet": (0, (6.0, 1.5, 1.5, 1.5))}
P5_STATE_SHORT = {"jet": "jet mode", "low_bh": rf"BH not grown ($\log M_{{\rm BH}} \leq {JET_LOGMBH:g}$)", "high_fedd": "radiative mode"}
_ALLQ_REF = [("SIMBA all quenched", dict(color="0.1", ls=(0, (1.2, 1.6)), lw=2.2, label=f"{M25_REF_LABEL}: all quenched"))] if SHOW_M25_ALLQ else []
_P5_NOTES = ["AGN state = SIMBA's jet criterion on the catalogue M_BH and f_Edd at the anchor;\n   in s50nox the jets act without X-ray heating, in s50noagn no feedback acts\n   (its 'jet mode' is a BH-growth label only)",
             f"bootstrap 16–84 % band only for populations of >= {P5_BAND_MIN} galaxies",
             M25_REF_NOTE]

# (a) per variant: the Part 3 contrasts and the grid split by state
P5_GRP, P5_ROWS = {}, {}
for v in VARIANTS:
    g = QM[QM["box"] == v].copy()
    if len(g) < P5_MIN_GAL:
        print(f"{v}: {len(g)} quenched galaxies under the mass cut — skipped"); continue
    P5_GRP[v] = agn_state_contrasts(v, g)
    groups = [(k, P5_GRP[v][k], dict(P3_STYLE[k], band=len(P5_GRP[v][k]) >= P5_BAND_MIN)) for k in P5_STATES if len(P5_GRP[v][k]) >= P5_MIN_GAL]
    print("  drawn: " + ", ".join(f"{k} (N={len(f)})" for k, f, _ in groups) + "; not drawn: " + (", ".join(f"{k} (N={len(P5_GRP[v][k])})" for k in P5_STATES if len(P5_GRP[v][k]) < P5_MIN_GAL) or "—"))
    _ref_save, M25_REF = M25_REF, _ALLQ_REF
    _, P5_ROWS[v] = ism_figure(f"paper_ism_prediction_agnstate_m50_{P5_SHORT.get(v, v)}", groups,
                               f"SIMBA {BOXES[v]['label']} quenched (catalogue),\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nsplit by the AGN state at the anchor",
                               extra_notes=_P5_NOTES, nbins=P5_NBIN, nmin=P5_NMIN)
    M25_REF = _ref_save

# (b) the same state across variants: colour = state, line style / marker = variant
P5X_GROUPS = [(f"{P5_SHORT.get(v, v)} {k}", P5_GRP[v][k], dict(color=P3_STYLE[k]["color"], ls=P5_VARIANT_LS.get(v, "-"), marker=BOXES[v]["marker"], lw=2.4, band=len(P5_GRP[v][k]) >= P5_BAND_MIN,
                                                              label=f"{P5_SHORT.get(v, v)}: {P5_STATE_SHORT[k]}"))
              for k in P5_CROSS_STATES for v in P5_GRP if len(P5_GRP[v][k]) >= P5_MIN_GAL]
if P5X_GROUPS:
    _ref_save, M25_REF = M25_REF, _ALLQ_REF
    DRAWN_P5X, ROWS_P5X = ism_figure("paper_ism_prediction_agnstate_variants_m50", P5X_GROUPS,
                                     f"SIMBA m50n512 feedback variants (catalogues), quenched:\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nthe same AGN state across the variants",
                                     extra_notes=_P5_NOTES, nbins=P5_NBIN, nmin=P5_NMIN)
    M25_REF = _ref_save

    # the same state under different physics: medians per variant with the Mann–Whitney p against the fiducial run's galaxies in that state
    _XV = (("lfdust", "log M_dust/M*"), ("lfh2", "log M_H2/M*"), ("log_mstar", "log M*"), ("age", "age [Gyr]"), ("lsig", "log Sigma_e"), ("lssfr", "log sSFR"),
           ("kappa_gas", "kappa_rot gas"), ("z", "z"), ("central", "central (1) / sat (0)"))
    for k in P5_CROSS_STATES:
        vs = [v for v in VARIANTS if v in P5_GRP and len(P5_GRP[v][k]) >= 2]
        if len(vs) < 2:
            continue
        ref = vs[0]
        print(f"\nstate '{k}' across the variants (medians; Mann–Whitney p against {ref} in the same state):")
        print(f"  {'':>16s} " + " ".join(f"{P5_SHORT.get(v, v):>22s}" for v in vs))
        for col, lab in _XV:
            a = P5_GRP[ref][k][col].dropna()
            cells = [f"{_stat(col)(a):7.3f} (n={len(a):4d})      "]
            for v in vs[1:]:
                b = P5_GRP[v][k][col].dropna()
                p = mannwhitneyu(a, b).pvalue if min(len(a), len(b)) >= 2 else np.nan
                cells.append(f"{_stat(col)(b):7.3f} (n={len(b):3d}) p={p:.1g}")
            print(f"  {lab:>16s} " + " ".join(f"{c:>22s}" for c in cells))

# (c) the scorecard: every population's offsets over the dust and the H2 panels, its dust-to-H2 ratio against the observed one
_det = OBS[(OBS["fdust_censor"] == "det") & (OBS["fh2_censor"] == "det")]
OBS_DUST_TO_H2 = float((_det["lfdust"] - _det["lfh2"]).median()) if len(_det) else np.nan


def offset_scorecard(pops, stem):
    """pops = [(name, frame, obs_vs_track rows of that track)] -> DataFrame ranked by the larger |median offset| of the dust / H2 panels; printed and written to <stem>_scorecard.csv."""
    print(f"\nscorecard: measured − track [dex] over the detections inside each track's x range, median over the dust panels and over the H2 panels;\n"
          f"  joint = the larger |median|; log M_dust/M_H2 = the population's median against the observed {OBS_DUST_TO_H2:+.2f} ({len(_det)} sources with both detected)")
    print(f"  {'population':22s} {'N gal':>6s} {'dust offset':>18s} {'H2 offset':>18s} {'joint':>6s} {'log Mdust/MH2':>14s} {'sat %':>6s} {'age':>5s} {'kappa_gas':>9s}")
    rows = []
    for name, g, O in pops:
        d = O[O["panel"].str.endswith("_fdust")]["offset_dex"]; h = O[O["panel"].str.endswith("_fh2")]["offset_dex"]
        md, mh = (d.median() if len(d) else np.nan), (h.median() if len(h) else np.nan)
        joint = np.nanmax([abs(md), abs(mh)]) if np.isfinite(md) or np.isfinite(mh) else np.nan
        rows.append(dict(population=name, n_gal=len(g), dust_offset=md, n_dust=len(d), h2_offset=mh, n_h2=len(h), joint=joint,
                         log_dust_to_h2=float((g["lfdust"] - g["lfh2"]).median()), sat_pct=100 * (1 - g["central"].mean()), age=g["age"].median(), kappa_gas=g["kappa_gas"].median()))
    if not rows:
        print("  (no population)"); return pd.DataFrame(rows)
    S = pd.DataFrame(rows).sort_values("joint")
    for _, r in S.iterrows():
        print(f"  {r['population']:22s} {r['n_gal']:6d} {r['dust_offset']:+7.2f} (N={r['n_dust']:2d})    {r['h2_offset']:+7.2f} (N={r['n_h2']:2d})    {r['joint']:5.2f} {r['log_dust_to_h2']:14.2f} {r['sat_pct']:6.0f} {r['age']:5.2f} {r['kappa_gas']:9.2f}")
    S.to_csv(os.path.join(OUT, f"{stem}_scorecard.csv"), index=False)
    print(f"  scorecard -> {os.path.join(OUT, f'{stem}_scorecard.csv')}")
    return S


_pop = []
if "ROWS_P4" in globals():
    for v, g, _ in P4_GROUPS:
        _pop.append((f"{P5_SHORT.get(v, v)} all", g, ROWS_P4[(ROWS_P4["kind"] == "obs_vs_track") & (ROWS_P4["track"] == v)]))
for v, rows in P5_ROWS.items():
    for k in P5_STATES:
        if len(P5_GRP[v][k]) >= P5_MIN_GAL:
            _pop.append((f"{P5_SHORT.get(v, v)} {k}", P5_GRP[v][k], rows[(rows["kind"] == "obs_vs_track") & (rows["track"] == k)]))
SCORE_P5 = offset_scorecard(_pop, "paper_ism_prediction_agnstate_variants_m50")

## Part 6 — the m25 AGN coupling classes for the other boxes

Parts 3–5 could only use the AGN state *at the anchor*. The classes of the m25 work (`powderday_flux_quenched_m25.ipynb`, `AGN_CLASSIFIER = "pre_threshold"`) are a statement about the Gyr *before* the galaxy started to quench, and everything they need is a catalogue column along the main progenitor branch — `masses.stellar`, `sfr`, `masses.gas`, `masses.bh`, `bh_fedd` at every snapshot from the anchor back to the m25 `end` snapshot (`END_SNAP`, the 9 % cosmic-age rule). The 100 and 50 Mpc catalogues ship the merger trees (`tree_data/progen_galaxy_star`, most massive progenitor at the previous snapshot); the 25 Mpc box uses the sidecar trees the m25 pipeline built with `caesar progen` (`output/cis25/progen_links/`). This part re-implements the rule verbatim, with the same constants and the same `find_quenching_times`:

1. **Histories** — for every galaxy of `Q` at every anchor whose catalogue *ladder* is complete (all snapshots from `END_SNAP[anchor]` to the anchor on disk; the missing ones are listed), one pass over the snapshots reads each catalogue once and follows every anchor's chains at the same time $\to$ `ism_prediction_histories_<box>.hdf5` (one group per anchor: `snaps`, `z`, `t_yr`, `gal_ids`, and the five `(n_snap, n_gal)` arrays; incremental per anchor, `OVERWRITE_HIST`).
2. **SFT / QT** — `find_quenching_times` on the instantaneous sSFR history (`sfr / masses.stellar`, strictly positive snapshots, $\geq 5$ of them; the last surviving event): SFT = the crossing below $1/t$, QT = the subsequent crossing below $0.2/t$ with the persistence check.
3. **$w_{\rm pre}$** — the ungated jet weight $w_{\rm jet} = {\rm clip}(\log_{10}(0.2/f_{\rm Edd}),\,0,\,1)$ for $\log M_{\rm BH} > 7.5$ (0 below, NaN where the chain is broken), averaged over the snapshots in $[t_{\rm SFT} - 1\,{\rm Gyr},\ t_{\rm SFT}]$ (nearest finite snapshot when the window falls before the tracked history); the onset = first snapshot with $x_{\rm coup} = w_{\rm jet}\,[f_{\rm gas} < 0.2] \geq 0.5$.
4. **Classes** (`pre_threshold`) — `weak`: $w_{\rm pre} < 0.10$, `strong`: $w_{\rm pre} \geq 0.50$, `intermediate` between; `no_AGN` when no finite $w_{\rm jet}$ exists, `no_event` when no SFT/QT pair is found. Fixed cuts, so the class fractions are free to differ between boxes (in m25: 87 weak / 119 intermediate / 32 strong / 28 no_event of 266). **`CLASS_SCHEME = "two"`** (default): only weak and strong are used — the rule's intermediate galaxies are redistributed by $w_{\rm pre}$ at `TWO_CLASS_THR` $= 0.30$ (the middle of the band; `coupling_class` in Part 0); the rule's three-class label is kept as `agn_class3`, and the cis25 check below compares *that* with the m25 pipeline. `"three"` restores the rule.

The classes are merged into `Q` / `QM` (`agn_class`, `w_pre`, `t_sft`, `t_qt`, `agn_onset`) and written to `ism_prediction_agn_classes.csv`. For the 25 Mpc box the m25 selection table (`powderday_quenched_selection_pt.fits`, `agn_class_pt`) is merged as `agn_class_m25` and compared galaxy by galaxy with the re-implementation — the check that this notebook's classes *are* the m25 classes. The second cell draws, per box, the figure-H grid split by class (the m25 reference's own class tracks — the whole-galaxy truth of the m25 sample under the same rule — as the reference), the weak and strong classes across the boxes, and the joint dust + H$_2$ scorecard of Part 5 extended with every (box, class) population.

In [ ]:
# ── Part 6a — the m25 AGN coupling classes for the other boxes: main-branch histories from the catalogues, SFT / QT, w_pre, the pre_threshold rule ──
H_BOXES = [b for b in os.environ.get("ISM_H_BOXES", "cis100,cis50,cis25,cis50nox").split(",") if b in BOXES]   # boxes whose histories are built (env ISM_H_BOXES narrows a test run)
OVERWRITE_HIST = False
XRAY_FGAS_MAX, PRE_JET_WINDOW_GYR, ONSET_THR, GYR = 0.2, 1.0, 0.5, 1e9   # powderday_flux_quenched_m25.ipynb cell "2" (PRE_THR_WEAK / PRE_THR_STRONG and the class scheme: Part 0)
END_SNAP = {134: 36, 125: 31, 116: 27, 110: 24, 105: 22, 100: 20, 95: 17, 90: 15, 83: 12, 78: 10}             # m25 TRACK_AGE_FRAC = 0.09: the end of the tracked history per anchor
HIST_COLS = {"mstar": "galaxy_data/dicts/masses.stellar", "sfr": "galaxy_data/sfr", "mgas": "galaxy_data/dicts/masses.gas", "mbh": "galaxy_data/dicts/masses.bh", "fedd": "galaxy_data/bh_fedd"}
HIST_FILE = os.path.join(OUT, "ism_prediction_histories_{box}.hdf5")
CLASS_CSV = os.path.join(OUT, "ism_prediction_agn_classes.csv")

try:
    from simbanator.analysis.quenching import find_quenching_times
except Exception:                       # no simbanator environment: load the module file alone (it needs numpy / scipy only)
    import importlib.util
    _spec = importlib.util.spec_from_file_location("simbanator_quenching", os.path.join(os.getcwd(), "simbanator", "analysis", "quenching.py"))
    _mod = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
    find_quenching_times = _mod.find_quenching_times


def progen_main(box, snap):
    """Index at snap-1 of the most massive progenitor of every galaxy at `snap` (-1 = none): the catalogue's tree_data, else the sidecar written by caesar progen; None if neither exists."""
    cat = cat_path(box, snap)
    for path in (cat, os.path.join(os.getcwd(), "output", box, "progen_links", os.path.basename(cat))):
        if os.path.exists(path):
            with h5py.File(path, "r") as f:
                if "tree_data/progen_galaxy_star" in f:
                    p = f["tree_data/progen_galaxy_star"][:]
                    return (p[:, 0] if p.ndim == 2 else p).astype(np.int64)
    return None


def ladder_missing(box, snap):
    return [s for s in range(END_SNAP[snap], snap + 1) if not os.path.exists(cat_path(box, s))]


def _read_hist(box, snap):
    """The five history columns of one catalogue (a dataset that does not exist — no BH yet at early snapshots — is NaN, as in the m25 pipeline) + z + the missing keys."""
    with h5py.File(cat_path(box, snap), "r") as f:
        n = f["galaxy_data/dicts/masses.stellar"].shape[0] if "galaxy_data/dicts/masses.stellar" in f else 0   # no galaxies yet -> empty catalogue
        cols = {k: (f[p][:].astype(float) if p in f else np.full(n, np.nan)) for k, p in HIST_COLS.items()}
        z = float(f["simulation_attributes"].attrs["redshift"])
        miss = [k for k, p in HIST_COLS.items() if p not in f] if n else ["(no galaxies)"]
    return cols, z, miss


def build_histories(box):
    """One pass over the snapshots (each catalogue read once): the chains of every anchor of `box` whose ladder is complete -> HIST_FILE[box] (one group per anchor)."""
    path = HIST_FILE.format(box=box)
    done = []
    if os.path.exists(path) and not OVERWRITE_HIST:
        with h5py.File(path, "r") as f:
            done = [int(k[4:]) for k in f.keys() if k.startswith("snap")]
    pool = Q[Q["box"] == box]
    todo = []
    for zt, s in ANCHORS.items():
        if not (pool["snap"] == s).any() or s in done:
            continue
        miss = ladder_missing(box, s)
        if miss:
            print(f"  {box} anchor {s} (z = {zt:g}): ladder incomplete, {len(miss)} catalogues missing ({miss[0]}–{miss[-1]}) — not built")
        else:
            todo.append(s)
    if not todo:
        return done
    smax, smin = max(todo), min(END_SNAP[s] for s in todo)
    print(f"  {box}: building anchors {todo} from {sum((pool['snap'] == s).sum() for s in todo)} galaxies of Q; snapshots {smax} -> {smin} ({smax - smin + 1} catalogues)")
    state, t0, absent = {}, time.time(), {}
    for s in range(smax, smin - 1, -1):
        cols, z, miss = _read_hist(box, s)
        for k in miss:
            absent.setdefault(k, []).append(s)
        prog = progen_main(box, s)
        n_here = len(cols["mstar"])
        for a in todo:
            if s == a:
                ids = np.sort(pool.loc[pool["snap"] == a, "gal_id"].to_numpy(np.int64))
                state[a] = dict(ids=ids, idx=ids.astype(float), snaps=[], z=[], data={k: [] for k in HIST_COLS})
            st = state.get(a)
            if st is None or s < END_SNAP[a]:
                continue
            idx = st["idx"]; ok = np.isfinite(idx) & (idx >= 0) & (idx < n_here)
            ii = idx[ok].astype(int)
            for k in HIST_COLS:
                arr = np.full(len(idx), np.nan); arr[ok] = cols[k][ii]; st["data"][k].append(arr)
            st["snaps"].append(s); st["z"].append(z)
            nxt = np.full(len(idx), np.nan)
            if prog is not None:
                p = prog[ii]; nxt[ok] = np.where(p >= 0, p, np.nan)
            st["idx"] = nxt
        if (smax - s) % 10 == 9 or s == smin:
            print(f"    snap {s:3d} (z = {z:.2f}) done, {time.time() - t0:5.0f} s", flush=True)
    for k, ss in absent.items():
        print(f"    '{k}' absent from {len(ss)} catalogues (snapshots {min(ss)}–{max(ss)}): NaN there" + (" — every chain ends above them" if k == "(no galaxies)" else ""))
    with h5py.File(path, "a") as f:
        for a, st in state.items():
            g = f.require_group(f"snap{a:03d}")
            for k in list(g.keys()):
                del g[k]
            zs = np.array(st["z"]); g["snaps"] = np.array(st["snaps"], np.int32); g["z"] = zs; g["t_yr"] = COSMO.age(zs).value * GYR; g["gal_ids"] = st["ids"]
            for k in HIST_COLS:
                g[k] = np.vstack(st["data"][k])
            frac = np.isfinite(g["mstar"][:]).mean(axis=1)
            print(f"    anchor {a}: {len(st['ids'])} chains x {len(zs)} snapshots; chain alive at z = 2 / 4: {np.interp(2.0, zs, frac):.2f} / {np.interp(4.0, zs, frac):.2f}")
    print(f"  histories -> {path}")
    return sorted(set(done) | set(todo))


def classify_anchor(box, a):
    """The m25 rule on one anchor's histories -> DataFrame (box, snap, gal_id, t_sft, t_qt, tau_q, w_pre, n_pre, t_onset, agn_onset, agn_class3 = the rule's three
    classes, agn_class = the class under CLASS_SCHEME)."""
    with h5py.File(HIST_FILE.format(box=box), "r") as f:
        g = f[f"snap{a:03d}"]
        t, ids = g["t_yr"][:], g["gal_ids"][:]
        H = {k: g[k][:] for k in HIST_COLS}
    order = np.argsort(t); t_inc = t[order]
    with np.errstate(all="ignore"):
        ssfr = np.where(H["mstar"] > 0, H["sfr"] / H["mstar"], np.nan)
        fgas = np.where(H["mstar"] > 0, H["mgas"] / H["mstar"], np.nan)
        bh_ok = np.isfinite(H["mbh"]) & np.isfinite(H["fedd"])
        wjet = np.where(bh_ok, np.where(H["mbh"] > 10 ** JET_LOGMBH, np.clip(np.log10(JET_FEDD / np.clip(H["fedd"], 1e-12, None)), 0.0, 1.0), 0.0), np.nan)
        xcoup = np.where(np.isfinite(wjet) & np.isfinite(fgas), wjet * (fgas < XRAY_FGAS_MAX).astype(float), np.nan)
    out = []
    for j, gid in enumerate(ids):
        r = dict(box=box, snap=a, gal_id=int(gid), t_sft=np.nan, t_qt=np.nan, tau_q=np.nan, w_pre=np.nan, n_pre=0, t_onset=np.nan, agn_onset="no_event", agn_class3="no_event", agn_class="no_event")
        s_j = ssfr[:, j]; valid = np.isfinite(s_j) & (s_j > 0) & np.isfinite(t)
        if valid.sum() >= 5:
            tv, sv = t[valid], s_j[valid]; o = np.argsort(tv); tv, sv = tv[o], sv[o]
            tu, ui = np.unique(tv, return_index=True); su = sv[ui]
            if len(tu) >= 5:
                qts, sfts, _, _dbg = find_quenching_times(tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts)); r["t_qt"], r["t_sft"] = float(qts[k]), float(sfts[k]); r["tau_q"] = r["t_qt"] - r["t_sft"]
        if np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]):
            ws = wjet[order, j]; fin = np.isfinite(ws)
            win = (t_inc >= r["t_sft"] - PRE_JET_WINDOW_GYR * GYR) & (t_inc <= r["t_sft"]) & fin
            if not win.any() and fin.any():
                jj = np.where(fin)[0]; win = np.zeros_like(fin); win[jj[np.argmin(np.abs(t_inc[jj] - r["t_sft"]))]] = True
            if win.any():
                r["w_pre"], r["n_pre"] = float(np.nanmean(ws[win])), int(win.sum())
            cs = xcoup[order, j]; jj = np.where(np.isfinite(cs) & (cs >= ONSET_THR))[0]
            if len(jj):
                r["t_onset"] = float(t_inc[jj[0]]); r["agn_onset"] = "lead" if r["t_onset"] < r["t_sft"] else ("concurrent" if r["t_onset"] <= r["t_qt"] else "late")
            else:
                r["agn_onset"] = "never" if np.isfinite(cs).any() else "no_AGN"
            w = r["w_pre"]
            r["agn_class3"] = "no_AGN" if not np.isfinite(w) else rule_class(w)
            r["agn_class"] = "no_AGN" if not np.isfinite(w) else coupling_class(w)
        out.append(r)
    return pd.DataFrame(out)


# ── build / classify ──
_cls = []
for box in H_BOXES:
    if not (Q["box"] == box).any():
        print(f"{box}: no quenched galaxy in Q — skipped"); continue
    print(f"{box}:")
    built = build_histories(box)
    for a in built:
        _cls.append(classify_anchor(box, a))
CLS = pd.concat(_cls, ignore_index=True) if _cls else pd.DataFrame(columns=KEY + ["t_sft", "t_qt", "tau_q", "w_pre", "n_pre", "t_onset", "agn_onset", "agn_class3", "agn_class"])
CLS["t_sft"], CLS["t_qt"], CLS["tau_q"] = CLS["t_sft"] / GYR, CLS["t_qt"] / GYR, CLS["tau_q"] / GYR      # Gyr in the tables
CLS["t_onset"] = CLS["t_onset"] / GYR
CLS.to_csv(CLASS_CSV, index=False)
print(f"\nclasses -> {CLASS_CSV} ({len(CLS)} galaxies; agn_class3 = the rule's three classes, agn_class = the {CLASS_SCHEME!r} scheme"
      + (f": weak < {TWO_CLASS_THR:g} <= strong, the rule's intermediate galaxies redistributed" if CLASS_SCHEME == "two" else "") + ")")

# the m25 pipeline's own labels for the 25 Mpc box: the check of the re-implementation, and the fallback when its histories were not built here
M25_CLS = None
if os.path.exists(M25_SELECTION):
    _t = Table.read(M25_SELECTION); _t = _t[np.asarray(_t["pop"]).astype(str) == "Q"]
    M25_CLS = pd.DataFrame({"box": "cis25", "snap": np.asarray(_t["snap"]).astype(int), "gal_id": np.asarray(_t["gal_id"]).astype(int),
                            "agn_class_m25": np.char.strip(np.asarray(_t["agn_class_pt"]).astype(str)), "w_pre_m25": np.asarray(_t["w_pre"], float)})
    M25_CLS["agn_class_m25s"] = [coupling_class(w) if np.isfinite(w) and c in CLASSES3 else c for w, c in zip(M25_CLS["w_pre_m25"], M25_CLS["agn_class_m25"])]   # under CLASS_SCHEME
    both = CLS.merge(M25_CLS, on=KEY, how="inner")
    if len(both):
        agree = (both["agn_class3"] == both["agn_class_m25"]).mean()
        print(f"\ncis25 check against the m25 pipeline's agn_class_pt (the rule's three classes): {len(both)} galaxies in common, labels agree for {100 * agree:.0f} %")
        print(pd.crosstab(both["agn_class3"].rename("this notebook"), both["agn_class_m25"].rename("m25 pipeline")))
        dw = (both["w_pre"] - both["w_pre_m25"]).dropna()
        print(f"  w_pre difference: median {dw.median():+.3f}, max |diff| {dw.abs().max():.3f} over {len(dw)} galaxies with both finite")
else:
    print(f"\nno m25 selection table at {M25_SELECTION}: the cis25 classes are this notebook's only")

for name in ("Q", "QM"):
    G = globals()[name]
    for c in ("t_sft", "t_qt", "tau_q", "w_pre", "t_onset", "agn_onset", "agn_class3", "agn_class", "agn_class_m25", "agn_class_m25s", "w_pre_m25"):
        if c in G.columns:
            G.drop(columns=c, inplace=True)
    G2 = G.merge(CLS[KEY + ["t_sft", "t_qt", "tau_q", "w_pre", "t_onset", "agn_onset", "agn_class3", "agn_class"]], on=KEY, how="left")
    if M25_CLS is not None:
        G2 = G2.merge(M25_CLS, on=KEY, how="left")
        fill = G2["agn_class"].isna() & G2["agn_class_m25"].notna()          # cis25 anchors without histories here: the m25 labels
        G2.loc[fill, "agn_class"] = G2.loc[fill, "agn_class_m25s"]; G2.loc[fill, "agn_class3"] = G2.loc[fill, "agn_class_m25"]; G2.loc[fill, "w_pre"] = G2.loc[fill, "w_pre_m25"]
    globals()[name] = G2
Q, QM = globals()["Q"], globals()["QM"]

print(f"\nAGN coupling classes ({CLASS_SCHEME}) of the tracks' sample (QM, log M* > {MASS_MIN:g}); per anchor: weak / intermediate / strong / other (no_AGN + no_event) / no history")
print(f"  {'box':10s} " + " ".join(f"{'z=' + str(zt):>15s}" for zt in ANCHORS) + f" {'all':>15s}")
for box in ALL_BOXES:
    g = QM[QM["box"] == box]
    if not len(g):
        continue
    def _cell(h):
        if not len(h): return f"{'—':>15s}"
        c = h["agn_class"].fillna("none").value_counts()
        return f"{c.get('weak', 0):3d}/{c.get('intermediate', 0):3d}/{c.get('strong', 0):3d}/{c.get('no_AGN', 0) + c.get('no_event', 0):2d}/{c.get('none', 0):3d}"
    print(f"  {box:10s} " + " ".join(_cell(g[g["snap"] == s]) for zt, s in ANCHORS.items()) + " " + _cell(g))
for box in ALL_BOXES:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES)]
    if len(g):
        print(f"  {box:10s} classified {len(g):4d}: " + ", ".join(f"{c} {100 * (g['agn_class'] == c).mean():.0f} %" for c in CLASSES)
              + f"; median w_pre {g['w_pre'].median():.2f}; tau_q {g['tau_q'].median():.2f} Gyr; onset lead {100 * (g['agn_onset'] == 'lead').mean():.0f} % (of {int(g['agn_onset'].notna().sum())} with a history here)")

In [ ]:
# ── Part 6b — the figure-H grid per box split by the AGN coupling class, the classes across the boxes, the scorecard over every population ──
P6_MIN_GAL = 6
_W_LO, _W_HI = (PRE_THR_WEAK, PRE_THR_STRONG) if CLASS_SCHEME == "three" else (TWO_CLASS_THR, TWO_CLASS_THR)   # the cuts the drawn classes are read at
P6_STYLE = {"weak": dict(color=CLASS_COLOR["weak"], ls="-", marker="o", lw=2.8, label=rf"weak AGN coupling ($w_{{\rm pre}} < {_W_LO:g}$)"),
            "intermediate": dict(color=CLASS_COLOR["intermediate"], ls=(0, (4.0, 1.6)), marker="s", lw=2.8, label=rf"intermediate ({PRE_THR_WEAK:g} $\leq w_{{\rm pre}} <$ {PRE_THR_STRONG:g})"),
            "strong": dict(color=CLASS_COLOR["strong"], ls=(0, (1.5, 1.2)), marker="^", lw=2.8, label=rf"strong AGN coupling ($w_{{\rm pre}} \geq {_W_HI:g}$)")}
P6_BOX_LS = {"cis100": "-", "cis50": (0, (4.0, 1.6)), "cis25": (0, (1.5, 1.2)), "cis50nox": (0, (6.0, 1.5, 1.5, 1.5))}
P6_CROSS_CLASSES = ["weak", "strong"]
_P6_REF = ([("SIMBA all quenched", dict(color="0.1", ls=(0, (1.2, 1.6)), lw=2.2, label=f"{M25_REF_LABEL}: all quenched"))] if SHOW_M25_ALLQ else []) + \
          [(f"SIMBA {c}", dict(color=CLASS_COLOR[c], ls="-", lw=1.1, label=f"{M25_REF_LABEL}: {c} AGN coupling")) for c in P6_CROSS_CLASSES]
_P6_NOTES = [f"AGN coupling class = the m25 pre_threshold rule on the catalogue histories:\n   w_pre = mean of clip(log10({JET_FEDD:g} / f_Edd), 0, 1) [log M_BH > {JET_LOGMBH:g}] over\n   [t_SFT − {PRE_JET_WINDOW_GYR:g} Gyr, t_SFT] along the main progenitor branch",
             (f"two classes: the rule's intermediate galaxies ({PRE_THR_WEAK:g} <= w_pre < {PRE_THR_STRONG:g}) redistributed at w_pre = {TWO_CLASS_THR:g}" if CLASS_SCHEME == "two"
              else f"the rule's three classes: weak < {PRE_THR_WEAK:g}, strong >= {PRE_THR_STRONG:g}, intermediate between"),
             f"bootstrap 16–84 % band only for populations of >= {P5_BAND_MIN} galaxies",
             M25_REF_NOTE + "; its own classes, same rule"]
_VARS6 = _VARS + (("w_pre", "w_pre"), ("tau_q", "tau_q [Gyr]"))


def contrast_table(title, grp, ref, cols=_VARS6):
    """Medians per group with the Mann–Whitney p against `ref` (the central flag: the central fraction)."""
    keys = [ref] + [k for k in grp if k != ref]
    print(f"\n{title}: " + ", ".join(f"{k} {len(grp[k])}" for k in keys) + f" (medians; Mann–Whitney p against {ref}):")
    print(f"  {'':>16s} " + " ".join(f"{k:>22s}" for k in keys))
    for col, lab in cols:
        if col not in grp[ref].columns:
            continue
        a = grp[ref][col].dropna()
        cells = [f"{_stat(col)(a):7.3f} (n={len(a):4d})      " if len(a) else f"{'—':>22s}"]
        for k in keys[1:]:
            b = grp[k][col].dropna()
            p = mannwhitneyu(a, b).pvalue if min(len(a), len(b)) >= 2 else np.nan
            cells.append(f"{_stat(col)(b):7.3f} (n={len(b):3d}) p={p:.1g}" if len(b) else f"{'—':>22s}")
        print(f"  {lab:>16s} " + " ".join(f"{c:>22s}" for c in cells))


P6_GRP, P6_ROWS = {}, {}
for box in [b for b in ALL_BOXES if (QM["box"] == b).any()]:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES)]
    if len(g) < P6_MIN_GAL:
        print(f"{box}: {len(g)} classified galaxies — no class figure"); continue
    nb, nm = (NBIN_X, NMIN_X) if len(g) >= 300 else (P4_NBIN, P4_NMIN)
    grp = {c: g[g["agn_class"] == c] for c in CLASSES}
    contrast_table(f"{box} AGN coupling classes", grp, "strong")
    groups = [(c, grp[c], dict(P6_STYLE[c], band=len(grp[c]) >= P5_BAND_MIN)) for c in CLASSES if len(grp[c]) >= P6_MIN_GAL]
    _ref_save, M25_REF = M25_REF, _P6_REF
    _, P6_ROWS[box] = ism_figure(f"paper_ism_prediction_agnclass_{box}", groups,
                                 f"SIMBA {BOXES[box]['label']} quenched (catalogue),\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nsplit by the m25 AGN coupling class (pre-SFT jet weight)",
                                 extra_notes=_P6_NOTES, nbins=nb, nmin=nm)
    M25_REF = _ref_save
    P6_GRP[box] = grp

# the same class across the boxes: colour = class, line style / marker = box
P6X_GROUPS = [(f"{box} {c}", P6_GRP[box][c], dict(color=CLASS_COLOR[c], ls=P6_BOX_LS.get(box, "-"), marker=BOXES[box]["marker"], lw=2.4, band=len(P6_GRP[box][c]) >= P5_BAND_MIN,
                                                 label=f"{BOXES[box]['label'].split(' (')[0]}{' ' + box[5:] if len(box) > 5 else ''}: {c}"))
              for c in P6_CROSS_CLASSES for box in P6_GRP if len(P6_GRP[box][c]) >= P6_MIN_GAL]
if P6X_GROUPS:
    _ref_save, M25_REF = M25_REF, _P6_REF
    DRAWN_P6X, ROWS_P6X = ism_figure("paper_ism_prediction_agnclass_boxes", P6X_GROUPS,
                                     f"SIMBA boxes (catalogues), quenched:\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\nthe weak and strong AGN coupling classes of every box",
                                     extra_notes=_P6_NOTES, nbins=P4_NBIN, nmin=P4_NMIN)
    M25_REF = _ref_save
    for c in P6_CROSS_CLASSES:
        grp = {box: P6_GRP[box][c] for box in P6_GRP if len(P6_GRP[box][c]) >= 2}
        if len(grp) >= 2:
            contrast_table(f"class '{c}' across the boxes", grp, list(grp)[0])

# the scorecard of Part 5 extended with every (box, class) population
_pop = [(f"{box} all", QM[QM["box"] == box], ROWS_BOXES[(ROWS_BOXES["kind"] == "obs_vs_track") & (ROWS_BOXES["track"] == box)]) for box, _, _ in BOX_GROUPS]
if "ROWS_P4" in globals():
    _pop += [(f"{P5_SHORT.get(v, v)} all", g, ROWS_P4[(ROWS_P4["kind"] == "obs_vs_track") & (ROWS_P4["track"] == v)]) for v, g, _ in P4_GROUPS if v not in [b for b, _, _ in BOX_GROUPS]]
for v, rows in P5_ROWS.items():
    _pop += [(f"{P5_SHORT.get(v, v)} {k}", P5_GRP[v][k], rows[(rows["kind"] == "obs_vs_track") & (rows["track"] == k)]) for k in P5_STATES if len(P5_GRP[v][k]) >= P5_MIN_GAL]
for box, rows in P6_ROWS.items():
    _pop += [(f"{box} {c}", P6_GRP[box][c], rows[(rows["kind"] == "obs_vs_track") & (rows["track"] == c)]) for c in CLASSES if len(P6_GRP[box][c]) >= P6_MIN_GAL]
SCORE_P6 = offset_scorecard(_pop, "paper_ism_prediction_agnclass_boxes")

## Part 7 — the dusty tail: does any quenched galaxy reach the observed dust fractions, or does the running median erase them?

Every track above is a running *median*: a population whose typical quenched galaxy holds $M_{\rm dust}/M_\star \sim 10^{-4.7}$ draws a track 1.6 dex under the ALMA-C11 detections even when a minority of its galaxies sits right on them. This part looks at the galaxies themselves:

* **(a) the distribution** — per population (box, AGN state at the anchor, coupling class) the percentiles of $\log M_{\rm dust}/M_\star$ and the number / fraction at or above each of `P7_THR` (the ALMA-C11 dust detections span $-3.8$ to $-2.6$, median $-3.0$) $\to$ `paper_ism_prediction_dusty_tail.csv`;
* **(b) what the tail is** — the galaxies at or above `P7_TAIL` ($\log M_{\rm dust}/M_\star \geq -3.2$) against the rest of the same box: the Part 6b contrasts (mass, age, $\Sigma_{\rm e}$, sSFR, H$_2$ fraction, rotation, redshift, central fraction, $w_{\rm pre}$, gas mass and particle number) and the AGN-state / class / anchor mix of the two groups;
* **(c) the figure** `paper_ism_prediction_dusty_tail` — the $M_{\rm dust}/M_\star$ row of the grid, one row per box (`P7_FIG_BOXES`), with **every galaxy as a point**, the tail in the box colour, the running median of Parts 2–6 together with the running 84th and 95th percentiles (`P7_QUANT`, same bins), and the observed points as before. `P7_Y = "fh2"` does the same for the H$_2$ fraction.

Needs Part 6b (`contrast_table`); the class and state columns are used where they exist.

In [ ]:
# ── Part 7 — the dusty tail: the individual galaxies behind the running medians; who reaches the observed dust fractions ──
P7_Y         = "fdust"                # the expensive quantity looked at (fdust | fh2)
P7_THR       = [-3.5, -3.2, -3.0]     # log10 thresholds: the number / fraction of galaxies at or above each is tabulated
P7_TAIL      = -3.2                   # the "tail": log f_dust at or above this (the ALMA-C11 dust detections span -3.8 to -2.6, median -3.0; 8 of the 9 are above -3.2)
P7_QUANT     = [0.84, 0.95]           # the running upper quantiles drawn with the median (same bins)
P7_FIG_BOXES = [b for b in ["cis100", "cis50", "cis25"] if (QM["box"] == b).sum() >= 10]   # rows of the figure
P7_MIN_TAIL  = 5                      # a tail / population smaller than this is not contrasted / tabulated
P7_XKEYS     = [x for x, y in GRID[0]]   # the cheap observables (age, Sigma_e, sSFR)
_ys7 = AXES[P7_Y]; _ycol = _ys7["col"]
_det7 = OBS.loc[OBS[_ys7["cens"]] == "det", _ycol].dropna()
print(f"observed {_ys7['short']} detections: N = {len(_det7)}, log range {_det7.min():.2f} to {_det7.max():.2f}, median {_det7.median():.2f}; the tail = log {_ys7['short']} >= {P7_TAIL:g}")


def run_quantile(x, y, q, nbins=NBIN_X, nmin=NMIN_X):
    """Running quantile q of y in equal-count x bins (plot space, the Part 2 binning) -> (x, y, n)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    nb = min(nbins, len(x) // nmin)
    if nb < 2:
        return pd.DataFrame(columns=["x", "y", "n"])
    e = np.quantile(x, np.linspace(0, 1, nb + 1)); ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb - 1)
    return pd.DataFrame([(np.median(x[ib == b]), np.quantile(y[ib == b], q), int((ib == b).sum())) for b in range(nb)], columns=["x", "y", "n"])


# (a) the distribution per population: percentiles of log f_dust and the number / fraction at or above each threshold
def tail_stats(name, g):
    v = g[_ycol].dropna().to_numpy(float)
    r = dict(population=name, n_gal=len(g), n_finite=len(v))
    if len(v):
        for p in (50, 84, 95, 99, 100):
            r[f"p{p}"] = float(np.percentile(v, p))
        for t in P7_THR:
            r[f"n_ge_{t:g}"] = int((v >= t).sum()); r[f"frac_ge_{t:g}"] = float((v >= t).mean())
    return r


_pops = []
for box in [b for b in ALL_BOXES if (QM["box"] == b).any()]:
    g = QM[QM["box"] == box]
    _pops.append((f"{box} all", g))
    if "agn_state" in g.columns:
        _pops += [(f"{box} {k}", g[g["agn_state"] == k]) for k in ("jet", "low_bh", "high_fedd") if (g["agn_state"] == k).sum() >= P7_MIN_TAIL]
    if "agn_class" in g.columns:
        _pops += [(f"{box} {c}", g[g["agn_class"] == c]) for c in CLASSES if (g["agn_class"] == c).sum() >= P7_MIN_TAIL]
TAIL = pd.DataFrame([tail_stats(n, g) for n, g in _pops])
print(f"\nlog {_ys7['short']} of the quenched galaxies under the mass cut, per population: percentiles and the number (fraction) at or above each threshold")
print(f"  {'population':22s} {'N':>6s} {'p50':>6s} {'p84':>6s} {'p95':>6s} {'p99':>6s} {'max':>6s}  " + "  ".join(f"{'>= ' + f'{t:g}':>15s}" for t in P7_THR))
for _, r in TAIL.iterrows():
    if r["n_finite"] == 0:
        continue
    print(f"  {r['population']:22s} {int(r['n_gal']):6d} {r['p50']:6.2f} {r['p84']:6.2f} {r['p95']:6.2f} {r['p99']:6.2f} {r['p100']:6.2f}  "
          + "  ".join(f"{int(r[f'n_ge_{t:g}']):5d} ({100 * r[f'frac_ge_{t:g}']:5.1f} %)" for t in P7_THR))
TAIL.to_csv(os.path.join(OUT, "paper_ism_prediction_dusty_tail.csv"), index=False)
print(f"  -> {os.path.join(OUT, 'paper_ism_prediction_dusty_tail.csv')}")

# (b) what the tail is: the galaxies at or above P7_TAIL against the rest of the same box, + their AGN state / class / anchor mix
_VARS7 = _VARS6 + (("lmgas", "log M_gas"), ("lfgas", "log M_gas/M*"), ("ngas", "N gas particles"))
for box in [b for b in ALL_BOXES if (QM["box"] == b).any()]:
    g = QM[QM["box"] == box].copy()
    with np.errstate(divide="ignore", invalid="ignore"):
        g["lmgas"] = np.log10(g["mgas"].to_numpy(float)); g["lfgas"] = np.log10(g["mgas"].to_numpy(float) / g["mstar"].to_numpy(float))
    tail, rest = g[g[_ycol] >= P7_TAIL], g[g[_ycol] < P7_TAIL]
    if len(tail) < P7_MIN_TAIL:
        print(f"\n{box}: {len(tail)} galaxies at or above log {_ys7['short']} = {P7_TAIL:g} — no contrast"); continue
    contrast_table(f"{box}: the dusty tail (log {_ys7['short']} >= {P7_TAIL:g}) vs the rest", {"tail": tail, "rest": rest}, "rest", cols=_VARS7)
    grp = np.where(g[_ycol] >= P7_TAIL, "tail", "rest")
    for col, lab in (("agn_state", "AGN state at the anchor"), ("agn_class", f"AGN coupling class ({CLASS_SCHEME})"), ("z_target", "anchor z")):
        if col in g.columns:
            ct = pd.crosstab(grp, g[col].fillna("none").astype(str), normalize="index") * 100
            print(f"  {lab}, % of each group: " + " | ".join(f"{k}: " + ", ".join(f"{c} {ct.loc[k, c]:.0f}" for c in ct.columns) for k in ("tail", "rest") if k in ct.index))

# (c) the figure: every galaxy as a point, the tail in the box colour, the running median + upper quantiles, the observed points
_nr, _nc = len(P7_FIG_BOXES), len(P7_XKEYS)
fig = plt.figure(figsize=(4.9 * _nc + 1.6, 4.1 * _nr + 2.9))
gs = GridSpec(_nr + 1, _nc, height_ratios=[1.0] * _nr + [0.46], hspace=0.10, wspace=0.08, figure=fig)
axes = [[fig.add_subplot(gs[i, j]) for j in range(_nc)] for i in range(_nr)]
_k = 0
for i, box in enumerate(P7_FIG_BOXES):
    g, st = QM[QM["box"] == box], BOXES[box]
    nb, nm = (NBIN_X, NMIN_X) if len(g) >= 300 else (P4_NBIN, P4_NMIN)
    for j, xk in enumerate(P7_XKEYS):
        ax, xs = axes[i][j], AXES[xk]
        ax.set_xscale("log" if xs["log"] else "linear"); ax.set_yscale("log" if _ys7["log"] else "linear"); ax.set_xlim(*xs["lim"]); ax.set_ylim(*_ys7["lim"])
        ok = np.isfinite(g[xs["col"]]) & np.isfinite(g[_ycol]); okt = ok & (g[_ycol] >= P7_TAIL)
        ax.scatter(_raw(xs, g.loc[ok, xs["col"]]), _raw(_ys7, g.loc[ok, _ycol]), s=5, c="0.72", alpha=0.45, lw=0, zorder=1, rasterized=True)
        ax.scatter(_raw(xs, g.loc[okt, xs["col"]]), _raw(_ys7, g.loc[okt, _ycol]), s=9, c=st["color"], alpha=0.75, lw=0, zorder=2, rasterized=True)
        tr = run_track(g[xs["col"]], g[_ycol], nbins=nb, nmin=nm)
        if len(tr):
            _draw_track(ax, tr, xs, _ys7, dict(st, lw=2.8), band=False, z=4)
        for q, ls in zip(P7_QUANT, [(0, (4.0, 1.6)), (0, (1.5, 1.2))]):
            tq = run_quantile(g[xs["col"]], g[_ycol], q, nbins=nb, nmin=nm)
            if len(tq):
                ax.plot(_raw(xs, tq["x"]), _raw(_ys7, tq["y"]), color=st["color"], lw=1.6, ls=ls, zorder=3, path_effects=[withStroke(linewidth=3.0, foreground="white")])
        ax.axhline(10 ** P7_TAIL if _ys7["log"] else P7_TAIL, color="0.3", lw=0.8, ls=":", zorder=0)
        draw_obs(ax, xs, _ys7)
        ax.text(0.03, 0.97, f"{st['label'].split(' (')[0]}: N = {int(ok.sum())}, {int(okt.sum())} ({100 * okt.sum() / max(int(ok.sum()), 1):.0f} %) at or above log {_ys7['short']} = {P7_TAIL:g}",
                transform=ax.transAxes, fontsize=FONT["note"], color="0.25", ha="left", va="top", zorder=9)
        ax.set_xlabel(xs["label"] if i == _nr - 1 else "", fontsize=FONT["label"]); ax.set_ylabel(_ys7["label"] if j == 0 else "", fontsize=FONT["label"])
        ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=FONT["tick"], labelbottom=(i == _nr - 1), labelleft=(j == 0)); ax.grid(False)
        for axis, spec in ((ax.yaxis, _ys7), (ax.xaxis, xs)):
            if spec["log"] and spec["lim"][0] >= 0.1 and spec["lim"][1] < 1e4:
                axis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:g}"))
        ax.set_title(f"({'abcdefghijkl'[_k]})", loc="left", fontsize=FONT["tag"], pad=6); _k += 1
flat = [a for row in axes for a in row]
sm = plt.cm.ScalarMappable(cmap=CM, norm=NORM); sm.set_array([])
cb = fig.colorbar(sm, ax=flat, pad=0.012, extend="max", fraction=0.02, aspect=40)
cb.set_label(AXES[OBS_COLOUR]["label"] + " of the observed galaxies", fontsize=FONT["label"]); cb.ax.tick_params(labelsize=FONT["tick"])
lax = fig.add_subplot(gs[_nr, :]); lax.axis("off")
_h = [(Line2D([], [], marker="o", ls="", ms=4, mfc="0.72", mec="none"), "every quenched galaxy under the cuts (one point per galaxy)"),
      (Line2D([], [], marker="o", ls="", ms=5, mfc="0.3", mec="none"), f"the dusty tail: log {_ys7['short']} >= {P7_TAIL:g} (box colour)"),
      (Line2D([], [], color="0.3", lw=2.8), "running median (the track of Parts 2–6)"),
      (Line2D([], [], color="0.3", lw=1.6, ls=(0, (4.0, 1.6))), f"running {100 * P7_QUANT[0]:.0f}th percentile (same bins)"),
      (Line2D([], [], color="0.3", lw=1.6, ls=(0, (1.5, 1.2))), f"running {100 * P7_QUANT[1]:.0f}th percentile"),
      (Line2D([], [], color="0.3", lw=0.8, ls=":"), f"log {_ys7['short']} = {P7_TAIL:g}"),
      (Line2D([], [], marker="o", ls="", ms=10, mfc=_cof(np.mean(OBS_CLIM)), mec=C_OBS_EDGE, mew=1.3), "observed, as Part 2 (colour = $A_V$, arrows = limits)")]
_leg = lax.legend([a for a, _ in _h], [b for _, b in _h], loc="upper left", bbox_to_anchor=(0.0, 0.78), ncol=2, frameon=False, fontsize=FONT["legend"], handlelength=2.2,
                  title=f"SIMBA quenched galaxies from the catalogues, z = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT.replace(chr(10), ' ')}:\nthe individual galaxies behind the running medians",
                  title_fontsize=FONT["legend"] + 0.5)
_leg._legend_box.align = "left"
paper_save(fig, "paper_ism_prediction_dusty_tail")
plt.show()

## Part 8 — three dust sides at $M_{\rm dust}/M_\star = 10^{-5}$ and $10^{-3.5}$: is the weak / strong AGN coupling class a predictor of the side?

Part 7 showed the quenched population is two-sided: a cloud at $M_{\rm dust}/M_\star \approx 10^{-3}$ and the bulk below $10^{-3.5}$. This part labels every quenched galaxy under the mass cut by its side (`P8_EDGES`): **dust-rich** at $\log M_{\rm dust}/M_\star \geq -3.5$ (where the ALMA-C11 detections sit), **undetected** in $[-5, -3.5)$ (the range of the observed upper limits), **no-dust** below $-5$ (zero dust included) — and asks whether the pre-quenching AGN coupling class (Part 6, `CLASS_SCHEME`) knows the side — box by box, where the classes exist (`P8_BOXES`):

* **(a) the class as a predictor** — the fraction of each class on each side with its 68 % Wilson interval, the class $\times$ side contingency, the $\chi^2$ $p$ and Cramér's $V$ of the association, the accuracy of the best "class $\Rightarrow$ side" rule against the majority baseline, and at each end (dust-rich against the rest, no-dust against the rest) Fisher's exact $p$ with the odds for weak relative to strong coupling $\to$ `paper_ism_prediction_dusty_split.csv` (the fractions), `..._stats.csv` (the association);
* **(b) $w_{\rm pre}$ against the other predictors** — for each quantity ($w_{\rm pre}$, the jet flag, age, sSFR, $M_{\rm gas}/M_\star$, $M_\star$, $\Sigma_{\rm e}$, $\kappa_{\rm rot}$, central / satellite, redshift, $\tau_{\rm q}$) the AUC (Mann–Whitney $U / n_1 n_0$) as a score for dust-rich against the rest and for no-dust against the rest (0.5 = no skill; $> 0.5$ a higher value favours that side), and the Spearman $\rho$ with the ordinal side (no-dust $<$ undetected $<$ dust-rich) $\to$ `paper_ism_prediction_dusty_split_predictors.csv`;
* **(c) the contrasts** — weak against strong *within* each side, and the two ends against the undetected middle *within* each class (Part 6b's `contrast_table`);
* **(d) the figures** — per box, the figure-H grid with the up-to-six (side $\times$ class) running medians (colour = class; solid = dust-rich, dashed = undetected, dotted = no-dust; the $M_{\rm dust}/M_\star$ row separates by construction, the H$_2$ row and the cheap observables are the test) $\to$ `paper_ism_prediction_dusty_split_<box>.*`, and a summary $\to$ `paper_ism_prediction_dusty_split_summary.*`: per box the side fractions of each class, and the AUC of every predictor at each end.

Needs Parts 3, 6 and 6b.

In [ ]:
# ── Part 8 — three dust sides at M_dust/M* = 1e-5 and 1e-3.5: does the weak / strong AGN coupling class predict the side? ──
from scipy.stats import fisher_exact, chi2_contingency, spearmanr
P8_EDGES   = (-5.0, -3.5)    # log10 M_dust/M*: "dust-rich" at or above -3.5 (the ALMA-C11 detections), "undetected" in [-5, -3.5), "no-dust" below -5 (zero dust included)
P8_SIDES   = ["dust-rich", "undetected", "no-dust"]
P8_MIN     = 6               # a (box, side, class) population smaller than this is listed, not drawn / contrasted
P8_PRED    = [("w_pre", "w_pre (AGN coupling)"), ("jet_f", "jet mode at anchor"), ("age", "stellar age"), ("lssfr", "log sSFR"), ("lfgas", "log M_gas/M*"),
              ("log_mstar", "log M*"), ("lsig", "log Sigma_e"), ("kappa_gas", "kappa_rot gas"), ("central", "central (vs sat)"), ("z", "redshift"), ("tau_q", "tau_q")]
P8_SIDE_LS = {"dust-rich": "-", "undetected": (0, (4.0, 1.6)), "no-dust": (0, (1.2, 1.5))}
P8_SIDE_MK = {"dust-rich": "o", "undetected": "s", "no-dust": "^"}
P8_ENDS    = [("dust-rich", "dust-rich vs the rest"), ("no-dust", "no-dust vs the rest")]     # the two one-against-the-rest contrasts of the predictor test
P8_ORD     = {"no-dust": 0, "undetected": 1, "dust-rich": 2}                                 # the ordinal side, for the rank correlation
with np.errstate(divide="ignore", invalid="ignore"):
    QM["lfgas"] = np.log10(QM["mgas"].to_numpy(float) / QM["mstar"].to_numpy(float))
QM["jet_f"] = QM["jet"].astype(float)
_fd8 = QM["fdust"].to_numpy(float)                                                           # the raw ratio: zero dust is "no-dust" (NaN on the log axis)
QM["side"] = np.where(_fd8 >= 10 ** P8_EDGES[1], "dust-rich", np.where(_fd8 < 10 ** P8_EDGES[0], "no-dust", "undetected"))
QM["side_ord"] = QM["side"].map(P8_ORD).astype(float)
P8_BOXES = [b for b in ALL_BOXES if (QM["box"] == b).any() and QM.loc[QM["box"] == b, "agn_class"].isin(CLASSES).sum() >= 2 * P8_MIN]
print(f"sides: dust-rich = log M_dust/M* >= {P8_EDGES[1]:g}, undetected = [{P8_EDGES[0]:g}, {P8_EDGES[1]:g}), no-dust < {P8_EDGES[0]:g} (zero dust included); "
      f"boxes with classified galaxies: {', '.join(P8_BOXES) or 'none (run Part 6)'}")
print(f"  {'box':10s} {'N':>5s} " + " ".join(f"{s:>18s}" for s in P8_SIDES) + f" {'zero dust':>9s} {'classified':>10s}")
for b in [b for b in ALL_BOXES if (QM['box'] == b).any()]:
    g = QM[QM["box"] == b]
    print(f"  {b:10s} {len(g):5d} " + " ".join(f"{int((g['side'] == s).sum()):6d} ({100 * (g['side'] == s).mean():5.1f} %)   " for s in P8_SIDES)
          + f" {int((g['fdust'] <= 0).sum()):9d} {int(g['agn_class'].isin(CLASSES).sum()):10d}")


def wilson(k, n, z=1.0):
    """Binomial fraction k/n with its Wilson interval at z sigma -> (p, lo, hi)."""
    if n == 0:
        return np.nan, np.nan, np.nan
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h


def auc_of(x, pos):
    """AUC of x as a score for the `pos` population against the rest (Mann–Whitney U / n1 n0): 0.5 = no skill, > 0.5 a higher x favours `pos` -> (auc, p, n_pos, n_rest)."""
    x, s = np.asarray(x, float), np.asarray(pos, bool); ok = np.isfinite(x)
    a, b = x[ok & s], x[ok & ~s]
    if min(len(a), len(b)) < 2:
        return np.nan, np.nan, len(a), len(b)
    r = mannwhitneyu(a, b)
    return float(r.statistic / (len(a) * len(b))), float(r.pvalue), len(a), len(b)


def cramers_v(tab):
    """Cramér's V of a contingency table (0 = no association, 1 = the row fixes the column)."""
    tab = np.asarray(tab, float); n = tab.sum()
    if n == 0 or min(tab.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(tab)[0]
    return float(np.sqrt(chi2 / (n * (min(tab.shape) - 1))))


# (a) the class as a predictor of the side; (b) w_pre against the other predictors
ROWS8, STAT8, PRED8 = [], [], []
for box in P8_BOXES:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES)].copy()
    tab = pd.crosstab(g["agn_class"], g["side"]).reindex(index=CLASSES, columns=P8_SIDES, fill_value=0)
    print(f"\n{box}: {len(g)} classified quenched galaxies; side fractions per class (68 % Wilson intervals):")
    print(f"  {'class':14s} {'N':>5s} " + " ".join(f"{s:>26s}" for s in P8_SIDES))
    for c in CLASSES + ["all classified"]:
        h = g if c == "all classified" else g[g["agn_class"] == c]
        cells = []
        for s in P8_SIDES:
            p = wilson(int((h["side"] == s).sum()), len(h))
            cells.append(f"{int((h['side'] == s).sum()):5d}  {100 * p[0]:5.1f} % [{100 * p[1]:5.1f}, {100 * p[2]:5.1f}]")
            ROWS8.append(dict(box=box, population=c, side=s, n=len(h), n_side=int((h["side"] == s).sum()), frac=p[0], frac_lo=p[1], frac_hi=p[2]))
        print(f"  {c:14s} {len(h):5d} " + " ".join(f"{x:>26s}" for x in cells))
    # the association: chi-square over the classes x sides table, Cramér's V, the best class -> side rule against the majority baseline,
    # and at each end the Fisher exact test of the 2 x 2 (class x [this side, the rest]) with the odds for weak relative to strong
    chi2, pc, _, _ = chi2_contingency(tab.to_numpy()) if (tab.to_numpy().sum(0) > 0).sum() >= 2 else (np.nan, np.nan, None, None)
    v = cramers_v(tab.to_numpy())
    acc = tab.to_numpy().max(1).sum() / len(g); base = tab.to_numpy().sum(0).max() / len(g)
    rule = ", ".join(f"{c} -> {tab.loc[c].idxmax()}" for c in CLASSES)
    st = dict(box=box, n=len(g), chi2_p=pc, cramers_v=v, rule=rule, rule_accuracy=acc, majority_baseline=base)
    print(f"  contingency (rows = class, columns = side):\n" + "\n".join(f"    {c:14s} " + " ".join(f"{tab.loc[c, s]:6d}" for s in P8_SIDES) for c in CLASSES))
    print(f"  chi-square p = {pc:.2g}, Cramér's V = {v:.2f}; the best rule '{rule}' is right for {100 * acc:.1f} % of the galaxies against a majority baseline of {100 * base:.1f} %")
    if len(CLASSES) == 2:
        for s, lab in P8_ENDS:
            t2 = np.array([[int((g["side"].eq(s) & g["agn_class"].eq(c)).sum()), int((~g["side"].eq(s) & g["agn_class"].eq(c)).sum())] for c in CLASSES])
            odds, pf = fisher_exact(t2)
            print(f"  {lab:24s}: Fisher exact p = {pf:.2g}; odds of being {s}, weak relative to strong = {odds:.2f}")
            st.update({f"fisher_p_{s}": pf, f"odds_{s}_weak_vs_strong": odds})
    STAT8.append(st)
    # the state × class × side mix, for the record
    if "agn_state" in g.columns:
        for c in CLASSES:
            h = g[g["agn_class"] == c]
            print(f"  {c:14s} AGN state at the anchor, dust-rich / no-dust fraction per state: "
                  + ", ".join(f"{k} {100 * (h.loc[h['agn_state'] == k, 'side'] == 'dust-rich').mean():.0f} / {100 * (h.loc[h['agn_state'] == k, 'side'] == 'no-dust').mean():.0f} % (n={int((h['agn_state'] == k).sum())})"
                              for k in ("jet", "low_bh", "high_fedd") if (h["agn_state"] == k).sum() >= 3))
    # (b) predictors: the AUC at each end (one side against the rest) and the Spearman rank correlation with the ordinal side
    print(f"  each quantity as a predictor: AUC for dust-rich vs the rest / no-dust vs the rest (0.5 = no skill; > 0.5: a higher value favours that side), "
          f"Spearman rho with the ordinal side (no-dust < undetected < dust-rich):")
    for col, lab in P8_PRED:
        if col not in g.columns:
            continue
        r = dict(box=box, predictor=col, label=lab)
        for s, _ in P8_ENDS:
            auc, p, n1, n0 = auc_of(g[col], g["side"] == s)
            r.update({f"auc_{s}": auc, f"p_{s}": p, f"n_{s}": n1})
        ok = np.isfinite(g[col].to_numpy(float))
        rho = spearmanr(g.loc[ok, col], g.loc[ok, "side_ord"]) if ok.sum() >= 3 and g.loc[ok, col].nunique() > 1 else None
        r.update(rho=float(rho.statistic) if rho is not None else np.nan, rho_p=float(rho.pvalue) if rho is not None else np.nan, n=int(ok.sum()),
                 skill=np.nanmax([abs(r[f"auc_{s}"] - 0.5) for s, _ in P8_ENDS]))
        PRED8.append(r)
    _pr = sorted([r for r in PRED8 if r["box"] == box], key=lambda r: -np.nan_to_num(r["skill"]))
    for r in _pr:
        print(f"    {r['label']:26s} AUC(dust-rich) {r['auc_dust-rich']:5.2f} p = {r['p_dust-rich']:.1g}   AUC(no-dust) {r['auc_no-dust']:5.2f} p = {r['p_no-dust']:.1g}   "
              f"rho = {r['rho']:+5.2f} p = {r['rho_p']:.1g}  (n = {r['n']})")
ROWS8, STAT8, PRED8 = pd.DataFrame(ROWS8), pd.DataFrame(STAT8), pd.DataFrame(PRED8)
ROWS8.to_csv(os.path.join(OUT, "paper_ism_prediction_dusty_split.csv"), index=False)
STAT8.to_csv(os.path.join(OUT, "paper_ism_prediction_dusty_split_stats.csv"), index=False)
PRED8.to_csv(os.path.join(OUT, "paper_ism_prediction_dusty_split_predictors.csv"), index=False)
print(f"\n  -> {os.path.join(OUT, 'paper_ism_prediction_dusty_split{,_stats,_predictors}.csv')}")

# (c) the contrasts: weak vs strong within each side, and the sides within each class (against the middle one)
_VARS8 = _VARS6 + (("lfgas", "log M_gas/M*"), ("ngas", "N gas particles"))
for box in P8_BOXES:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES)]
    for side in P8_SIDES:
        grp = {c: g[(g["side"] == side) & (g["agn_class"] == c)] for c in CLASSES}
        if all(len(v) >= P8_MIN for v in grp.values()):
            contrast_table(f"{box}, {side} side: the classes", grp, "strong", cols=_VARS8)
        else:
            print(f"\n{box}, {side} side: " + ", ".join(f"{c} {len(v)}" for c, v in grp.items()) + f" — fewer than {P8_MIN} in a class, no contrast")
    for c in CLASSES:
        grp = {s: g[(g["side"] == s) & (g["agn_class"] == c)] for s in P8_SIDES if len(g[(g["side"] == s) & (g["agn_class"] == c)]) >= P8_MIN}
        if len(grp) >= 2 and "undetected" in grp:
            contrast_table(f"{box}, {c} coupling: the sides", grp, "undetected", cols=_VARS8)

# (d) per box: the grid with the (side x class) tracks
_P8_NOTES = [f"sides at the anchor: dust-rich = log M_dust/M* >= {P8_EDGES[1]:g}, undetected = [{P8_EDGES[0]:g}, {P8_EDGES[1]:g}), no-dust < {P8_EDGES[0]:g} (zero dust included);\n"
             f"   colour = the AGN coupling class ({CLASS_SCHEME}), solid = dust-rich, dashed = undetected, dotted = no-dust",
             f"bootstrap 16–84 % band only for populations of >= {P5_BAND_MIN} galaxies", M25_REF_NOTE + "; its own classes, same rule"]
P8_ROWS = {}
for box in P8_BOXES:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES)]
    nb, nm = (NBIN_X, NMIN_X) if len(g) >= 300 else (P4_NBIN, P4_NMIN)
    groups = []
    for side in P8_SIDES:
        for c in CLASSES:
            f = g[(g["side"] == side) & (g["agn_class"] == c)]
            if len(f) >= P8_MIN:
                groups.append((f"{side} {c}", f, dict(color=CLASS_COLOR[c], ls=P8_SIDE_LS[side], marker=P8_SIDE_MK[side], lw=2.6, band=len(f) >= P5_BAND_MIN,
                                                     label=f"{side}: {c} AGN coupling")))
            else:
                print(f"  {box} {side} {c}: {len(f)} galaxies — not drawn")
    if len(groups) < 2:
        continue
    _ref_save, M25_REF = M25_REF, _P6_REF
    _, P8_ROWS[box] = ism_figure(f"paper_ism_prediction_dusty_split_{box}", groups,
                                 f"SIMBA {BOXES[box]['label']} quenched (catalogue),\nz = {min(ANCHORS):g}–{max(ANCHORS):g}, {SEL_TXT},\ndust sides at $M_{{\\rm dust}}/M_\\star = 10^{{{P8_EDGES[0]:g}}}, 10^{{{P8_EDGES[1]:g}}}$ x the AGN coupling class",
                                 extra_notes=_P8_NOTES, nbins=nb, nmin=nm)
    M25_REF = _ref_save

# (e) the summary: top row, per box, the side fractions per class; bottom row, the AUC of every predictor at each end
if len(P8_BOXES):
    _nb = len(P8_BOXES); _nc = max(2, _nb)
    fig = plt.figure(figsize=(4.6 * _nc + 1.0, 10.8))
    gs = fig.add_gridspec(2, 2 * _nc, height_ratios=[1.0, 1.15], hspace=0.55, wspace=1.2)
    _xs = np.arange(len(P8_SIDES)); _off = np.linspace(-0.16, 0.16, len(CLASSES))
    _ymax = 100 * ROWS8["frac_hi"].max() * 1.45
    for i, box in enumerate(P8_BOXES):
        ax = fig.add_subplot(gs[0, 2 * i:2 * i + 2])
        for k, c in enumerate(CLASSES):
            r = ROWS8[(ROWS8["box"] == box) & (ROWS8["population"] == c)].set_index("side").reindex(P8_SIDES)
            ax.errorbar(_xs + _off[k], 100 * r["frac"], yerr=[100 * (r["frac"] - r["frac_lo"]), 100 * (r["frac_hi"] - r["frac"])], fmt="o", color=CLASS_COLOR[c], ms=8,
                        capsize=3, lw=1.6, label=f"{c} AGN coupling (N = {int(r['n'].iloc[0])})", zorder=3)
        r = ROWS8[(ROWS8["box"] == box) & (ROWS8["population"] == "all classified")].set_index("side").reindex(P8_SIDES)
        ax.plot(_xs, 100 * r["frac"], ls="", marker="_", ms=22, mew=2.0, color="0.35", label=f"all classified (N = {int(r['n'].iloc[0])})", zorder=2)
        st = STAT8[STAT8["box"] == box].iloc[0]
        txt = f"chi-square p = {st['chi2_p']:.1g}, Cramér's V = {st['cramers_v']:.2f}\nbest rule {100 * st['rule_accuracy']:.0f} % vs base {100 * st['majority_baseline']:.0f} %"
        for s, lab in P8_ENDS:
            if f"fisher_p_{s}" in st and np.isfinite(st[f"fisher_p_{s}"]):
                txt += f"\n{s}: Fisher p = {st[f'fisher_p_{s}']:.1g}, odds weak/strong = {st[f'odds_{s}_weak_vs_strong']:.1f}"
        ax.text(0.0, -0.13, txt, transform=ax.transAxes, ha="left", va="top", fontsize=FONT["note"], color="0.3", zorder=4)   # the association, as a caption under the panel
        ax.set_xticks(_xs); ax.set_xticklabels(P8_SIDES, fontsize=FONT["tick"]); ax.set_xlim(-0.5, len(P8_SIDES) - 0.5)
        ax.set_ylim(0, _ymax); ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        if i == 0:
            ax.set_ylabel("fraction of the class on the side  [%]", fontsize=FONT["label"])
        ax.legend(loc="upper left", frameon=False, fontsize=FONT["legend"] - 1, title="68 % Wilson intervals", title_fontsize=FONT["legend"] - 1)
        ax.set_title(f"(a) {BOXES[box]['label'].split(' (')[0]}{'' if len(box) <= 5 else ' ' + box[5:]}: the class as a predictor of the side", loc="left", fontsize=FONT["tag"])
    _labs = [lab for col, lab in P8_PRED if col in PRED8["predictor"].values]
    _cols = [col for col, lab in P8_PRED if col in PRED8["predictor"].values]
    _yy = np.arange(len(_cols))[::-1]
    for j, (s, lab) in enumerate(P8_ENDS):
        ax = fig.add_subplot(gs[1, j * _nc:(j + 1) * _nc])
        for box in P8_BOXES:
            r = PRED8[PRED8["box"] == box].set_index("predictor").reindex(_cols)
            ax.plot(r[f"auc_{s}"], _yy, ls="", marker=BOXES[box]["marker"], color=BOXES[box]["color"], ms=8, mec="k", mew=0.5,
                    label=BOXES[box]["label"].split(" (")[0] + ("" if len(box) <= 5 else " " + box[5:]), zorder=3)
        ax.axvline(0.5, color="0.3", lw=1.0, ls=":"); ax.set_yticks(_yy); ax.set_yticklabels(_labs if j == 0 else [""] * len(_labs), fontsize=FONT["tick"])
        ax.set_ylim(-1.4, len(_cols) - 0.5)                                                        # an empty row at the bottom for the box legend
        ax.set_xlabel(f"AUC as a score for {s} against the rest\n(0.5 = no skill; > 0.5: a higher value favours {s})", fontsize=FONT["label"] - 1)
        ax.set_xlim(0.0, 1.0); ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        ax.legend(loc="lower left" if j == 0 else "lower right", frameon=False, fontsize=FONT["legend"], ncol=max(1, len(P8_BOXES)), columnspacing=1.0, handletextpad=0.4)
        ax.set_title(f"({'bc'[j]}) every quantity as a predictor: {lab}", loc="left", fontsize=FONT["tag"])
    paper_save(fig, "paper_ism_prediction_dusty_split_summary")
    plt.show()

## Part 9 — the X-ray channel after quenching: exposure, timing, and the class held fixed

Part 8 found the pre-quenching coupling class is no predictor of the dust content, while Parts 4–5 found the X-ray channel of the fiducial run is what removes the dust (s50 vs s50nox). The two are not in conflict: $w_{\rm pre}$ is the *ungated* jet weight before SFT (a BH-state label), whereas SIMBA's X-ray feedback fires only in jet mode **and** at $f_{\rm gas} < 0.2$. This part measures, for every classified galaxy, what the channel actually did to it after quenching, from the same catalogue histories (Part 6a: `mgas`, `mbh`, `fedd` along the main branch):

* **the exposure** $E_x = \int_{\rm SFT}^{\rm anchor} w_{\rm jet}\,[f_{\rm gas} < 0.2]\,{\rm d}t$ (Gyr; `e_x`), its duty $f_x = E_x / (t_{\rm anchor} - t_{\rm SFT})$, the gas-poor time $T_{\rm gas\text{-}poor} = \int [f_{\rm gas} < 0.2]\,{\rm d}t$ and the jet weight while gas-poor $r_x = E_x / T_{\rm gas\text{-}poor}$ (the gate held fixed), the ungated $E_{\rm jet}$, the same over $[{\rm SFT} - 1\,{\rm Gyr}, {\rm SFT}]$ (`e_x_pre`), the gate's timing $t_{\rm gate} - t_{\rm SFT}$ (first $f_{\rm gas} < 0.2$ from 1 Gyr before SFT on) and the **quenching sequence** (`gate_at_sft`: $f_{\rm gas}$ at SFT below 0.2 = gas-poor first, above = star formation stops first) $\to$ `ism_prediction_xray_exposure.csv`, merged into `Q` / `QM`;
* **the target** is the dust-to-gas ratio $\log M_{\rm dust}/M_{\rm gas}$ (`ldg`) next to $\log M_{\rm dust}/M_\star$ (`lfd`), zero dust at the floor `P9_FLOOR`: the gate is a gas-fraction cut, so the dust *fraction* correlates with exposure by construction; the dust-to-gas ratio does not, and it is the quantity the no-X-ray runs change (1.7 dex in $M_{\rm dust}/M_{\rm H_2}$, Part 5);
* **(a)** the exposure per class; **(a2)** the populations the histories define — the timing of the first X-ray episode (`agn_onset` of Part 6a: never / before SFT / in [SFT, QT] / after QT) and the sequence $\times$ class — with their dust content, disc, BH and environment $\to$ `paper_ism_prediction_xray_exposure_populations.csv`; **(b)** every quantity as a predictor (Spearman $\rho$ with the two targets, overall and inside each class; the AUC at the two dust ends) $\to$ `..._predictors.csv`; **(c)** the class with the exposure held fixed and the exposure with the class held fixed: $E_x$ terciles $\times$ class, and the same inside terciles of the time since SFT and of the gas-poor time (the confounders) $\to$ `..._stats.csv`; **(d)** contrasts: the never-exposed galaxies against the exposed, and the two sequences inside each class (`contrast_table`); **(e)** the figure $\to$ `paper_ism_prediction_xray_exposure.{png,pdf,csv}`: per box, $M_{\rm dust}/M_{\rm gas}$ against $E_x$ per class with the s50nox / s50noagn medians as the zero-exposure reference, against the time since SFT per onset population, against the gas-poor time per $r_x$ tercile, and the rank correlations of every predictor.

**Result (2026-08-30, cis100 / cis50 / cis25 catalogue histories).** The channel is nearly universal and saturates fast: 98–100 % of the classified quenched galaxies are exposed ($f_x \approx 0.9$, $r_x \approx 0.95$: once gas-poor the BH is in full jet mode), and the dust-to-gas ratio drops from the no-X-ray level ($-1.4$ in s50nox / s50noagn) to $-2.5$ within 0.25 Gyr of exposure and to $-3.5$ by 0.5 Gyr, then stays flat (a slow recovery to $-3.0$ over 4–8 Gyr). So the dose beyond the first half Gyr adds nothing ($\rho(E_x, {\rm D/G}) = -0.08$ overall in cis100, but $-0.40$ inside the first 0.8 Gyr since SFT, $-0.44$ inside the shortest gas-poor tercile), and at fixed gas-poor time the tercile with the lowest jet weight keeps $\sim 1$ dex more dust per unit gas. **The 39 cis100 galaxies never exposed** (no X-ray episode; 1.3 %) sit at $\log M_{\rm dust}/M_\star = -3.0$ — the observed value — with D/G $= -1.75$, none dust-free, $\kappa_{\rm rot}^{\rm gas} = 0.85$, age 2.9 Gyr: the s50nox phenotype inside the fiducial box. They are not strong-coupling discs with the gate closed: their BH never grew ($\log M_{\rm BH} = 6.4$ vs 8.3, 5 % in jet mode), 90 % are satellites, $\log M_\star = 10.5$, i.e. environmentally quenched galaxies whose BH never reached the jet regime (none in cis50, 4 in cis25). Among the exposed, the dust content is set by *when* the channel first fired, not for how long: first episode after QT $-2.6$ / before SFT $-3.1$ / during [SFT, QT] $-3.6$ (cis100), and by the sequence — gas-poor before SFT keeps 0.45–0.5 dex more D/G than star formation stopping first, inside both classes ($p \leq 2\times10^{-13}$ in cis100). The class enters only as a second-order modulation, and in the direction of the m25 figure G: at fixed exposure, sequence or gas-poor time the strong class keeps 0.15–0.5 dex more D/G than the weak one (cis100 $p = 10^{-4}$, cis50 $p = 0.01$; not in cis25, N = 7–18). The prediction that strong-$w_{\rm pre}$ galaxies quenching while still gas-rich escape the channel is **false**: they are exposed as soon as their gas drops (D/G $-3.5$, $\kappa = 0.41$).

Needs Parts 6a and 8 (`CLS`, the histories, `_VARS8`, `auc_of`, the sides).

In [ ]:
# ── Part 9 — the X-ray exposure after quenching: the gated channel (jet weight x [f_gas < 0.2]) integrated from SFT to the anchor, against the dust-to-gas ratio and the dust fraction; the class held fixed ──
from scipy.stats import kruskal
P9_BOXES  = [b for b in ["cis100", "cis50", "cis25"] if os.path.exists(HIST_FILE.format(box=b)) and (QM["box"] == b).any()]
P9_MIN    = 6                       # a population smaller than this is listed, not contrasted / drawn
P9_FLOOR  = -6.5                    # log M_dust/M* and log M_dust/M_gas of the zero-dust galaxies are drawn / binned at this floor (never dropped: they are the end point of the removal)
P9_EX_EDGES = [0.0, 0.0, 0.25, 0.5, 1.0, 2.0, 6.0]   # E_x bins [Gyr]: the first is E_x = 0 exactly (never exposed), then (0, 0.25], (0.25, 0.5], (0.5, 1], (1, 2], (2, 6]
P9_DG_REF = ["cis50nox", "cis50noagn"]              # the runs without the X-ray channel: their median dust-to-gas ratio drawn as the "zero exposure" reference
P9_PRED = [("e_x", "E_x = int w_jet [f_gas<0.2] dt, SFT->anchor [Gyr]", "$E_x$ (X-ray exposure)"), ("f_x", "f_x = E_x / (t_anchor - t_SFT)", "$f_x$ (X-ray duty since SFT)"),
           ("r_x", "r_x = E_x / T_gaspoor (jet weight while gas-poor)", "$r_x$ (jet weight while gas-poor)"), ("t_gaspoor", "T_gaspoor = int [f_gas<0.2] dt, SFT->anchor [Gyr]", "$T_{\\rm gas-poor}$"),
           ("e_jet", "E_jet = int w_jet dt, SFT->anchor [Gyr]", "$E_{\\rm jet}$ (ungated)"), ("dt_sft", "t_anchor - t_SFT [Gyr]", "time since SFT"),
           ("lag_gate", "t_gate - t_SFT [Gyr] (first f_gas < 0.2)", "$t_{\\rm gate} - t_{\\rm SFT}$"), ("e_x_pre", "E_x over [SFT - 1 Gyr, SFT]", "$E_x$ before SFT"),
           ("w_pre", "w_pre (ungated pre-SFT jet weight)", "$w_{\\rm pre}$ (the class)"), ("jet_f", "jet mode at the anchor", "jet mode at the anchor"),
           ("kappa_gas", "kappa_rot gas", "$\\kappa_{\\rm rot}$ gas"), ("age", "stellar age", "stellar age"), ("lssfr", "log sSFR", "log sSFR"), ("lfgas", "log M_gas/M* (anchor)", "log $M_{\\rm gas}/M_\\star$")]
P9_ONSET = [("never", "never exposed (no X-ray episode)", "0.05", "s"), ("lead", "first X-ray episode before SFT", "#fdae61", "o"), ("concurrent", "first episode in [SFT, QT]", "#d7191c", "o"),
            ("late", "first episode after QT", "#2c7bb6", "o")]                                  # agn_onset of Part 6a: the first snapshot with w_jet [f_gas < 0.2] >= ONSET_THR
P9_TARGETS = [("ldg", "log M_dust/M_gas"), ("lfd", "log M_dust/M*")]
_trapz = getattr(np, "trapezoid", None) or np.trapz


def exposure_anchor(box, a, cls):
    """The exposure integrals of one anchor's histories for the galaxies with SFT / QT in `cls` (indexed by gal_id, Gyr) -> DataFrame (box, snap, gal_id, t_anchor, dt_sft, dt_qt, e_x, e_jet, e_x_qt,
    t_gaspoor, f_x, f_jet, r_x, e_x_pre, e_jet_pre, lag_gate, gate_at_sft, fgas_sft, fgas_anchor). w_jet, f_gas and the gate as in classify_anchor; a missing BH counts as w_jet = 0."""
    with h5py.File(HIST_FILE.format(box=box), "r") as f:
        g = f[f"snap{a:03d}"]
        t, ids = g["t_yr"][:] / GYR, g["gal_ids"][:]
        H = {k: g[k][:] for k in HIST_COLS}
    order = np.argsort(t); t_inc = t[order]; t_anchor = float(t_inc[-1])
    with np.errstate(all="ignore"):
        fgas = np.where(H["mstar"] > 0, H["mgas"] / H["mstar"], np.nan)[order]
        bh_ok = np.isfinite(H["mbh"]) & np.isfinite(H["fedd"])
        wjet = np.where(bh_ok, np.where(H["mbh"] > 10 ** JET_LOGMBH, np.clip(np.log10(JET_FEDD / np.clip(H["fedd"], 1e-12, None)), 0.0, 1.0), 0.0), np.nan)[order]
    gate = np.where(np.isfinite(fgas), (fgas < XRAY_FGAS_MAX).astype(float), np.nan)
    xcoup = wjet * gate

    def integ(y, t0, t1):
        """int y dt over [t0, t1] on the snapshot grid (linear interpolation, NaN = 0: no BH / no galaxy = no feedback)."""
        if not (np.isfinite(t0) and np.isfinite(t1)) or t1 <= t0:
            return 0.0
        y = np.nan_to_num(y, nan=0.0)
        tt = np.concatenate(([t0], t_inc[(t_inc > t0) & (t_inc < t1)], [t1]))
        return float(_trapz(np.interp(tt, t_inc, y), tt))

    out = []
    for j, gid in enumerate(ids):
        gid = int(gid)
        r = dict(box=box, snap=a, gal_id=gid, t_anchor=t_anchor, dt_sft=np.nan, dt_qt=np.nan, e_x=np.nan, e_jet=np.nan, e_x_qt=np.nan, t_gaspoor=np.nan, f_x=np.nan, f_jet=np.nan,
                 r_x=np.nan, e_x_pre=np.nan, e_jet_pre=np.nan, lag_gate=np.nan, gate_at_sft=np.nan, fgas_sft=np.nan, fgas_anchor=np.nan)
        if gid not in cls.index:
            out.append(r); continue
        t_sft, t_qt = float(cls.loc[gid, "t_sft"]), float(cls.loc[gid, "t_qt"])
        if not (np.isfinite(t_sft) and np.isfinite(t_qt)):
            out.append(r); continue
        x, w, gt, fg = xcoup[:, j], wjet[:, j], gate[:, j], fgas[:, j]
        r["dt_sft"], r["dt_qt"] = t_anchor - t_sft, t_anchor - t_qt
        r["e_x"], r["e_jet"], r["e_x_qt"] = integ(x, t_sft, t_anchor), integ(w, t_sft, t_anchor), integ(x, t_qt, t_anchor)
        r["t_gaspoor"] = integ(gt, t_sft, t_anchor)
        r["e_x_pre"], r["e_jet_pre"] = integ(x, t_sft - PRE_JET_WINDOW_GYR, t_sft), integ(w, t_sft - PRE_JET_WINDOW_GYR, t_sft)
        if r["dt_sft"] > 0:
            r["f_x"], r["f_jet"] = r["e_x"] / r["dt_sft"], r["e_jet"] / r["dt_sft"]
        if r["t_gaspoor"] > 0:
            r["r_x"] = r["e_x"] / r["t_gaspoor"]
        fin = np.isfinite(fg)
        if fin.any():
            r["fgas_sft"] = float(np.interp(t_sft, t_inc[fin], fg[fin])); r["fgas_anchor"] = float(fg[fin][-1])
            r["gate_at_sft"] = float(r["fgas_sft"] < XRAY_FGAS_MAX)
            jj = np.where(fin & (fg < XRAY_FGAS_MAX) & (t_inc >= t_sft - PRE_JET_WINDOW_GYR))[0]      # the gate: first gas-poor snapshot from 1 Gyr before SFT on
            if len(jj):
                r["lag_gate"] = float(t_inc[jj[0]] - t_sft)
        out.append(r)
    return pd.DataFrame(out)


# ── the exposures of every classified anchor ──
EXP_CSV = os.path.join(OUT, "ism_prediction_xray_exposure.csv")
_exp = []
for box in P9_BOXES:
    with h5py.File(HIST_FILE.format(box=box), "r") as f:
        anchors = sorted(int(k[4:]) for k in f.keys() if k.startswith("snap"))
    for a in anchors:
        cls = CLS[(CLS["box"] == box) & (CLS["snap"] == a)].set_index("gal_id")[["t_sft", "t_qt"]]
        _exp.append(exposure_anchor(box, a, cls))
EXP = pd.concat(_exp, ignore_index=True) if _exp else pd.DataFrame(columns=KEY)
EXP.to_csv(EXP_CSV, index=False)
_EXP_COLS = [c for c in EXP.columns if c not in KEY]
for name in ("Q", "QM"):
    G = globals()[name]
    G.drop(columns=[c for c in _EXP_COLS if c in G.columns], inplace=True)
    globals()[name] = G.merge(EXP, on=KEY, how="left")
Q, QM = globals()["Q"], globals()["QM"]
with np.errstate(divide="ignore", invalid="ignore"):
    for G in (Q, QM):
        _fd, _mg, _md = G["fdust"].to_numpy(float), G["mgas"].to_numpy(float), G["mdust"].to_numpy(float)
        G["lfd"] = np.where(_fd > 0, np.log10(np.where(_fd > 0, _fd, 1.0)), P9_FLOOR)                            # zero dust at the floor (the removal's end point)
        G["ldg"] = np.where((_md > 0) & (_mg > 0), np.log10(np.where((_md > 0) & (_mg > 0), _md / np.where(_mg > 0, _mg, 1.0), 1.0)), np.where(_mg > 0, P9_FLOOR, np.nan))
        if "lfgas" not in G.columns:
            G["lfgas"] = np.log10(_mg / G["mstar"].to_numpy(float))
        if "jet_f" not in G.columns:
            G["jet_f"] = G["jet"].astype(float)
        if "side" not in G.columns:
            G["side"] = np.where(_fd >= 10 ** P8_EDGES[1], "dust-rich", np.where(_fd < 10 ** P8_EDGES[0], "no-dust", "undetected"))
print(f"exposures -> {EXP_CSV} ({int(np.isfinite(EXP['e_x']).sum())} galaxies with SFT / QT over {', '.join(P9_BOXES)}); the clock starts at SFT, the gate is f_gas < {XRAY_FGAS_MAX:g} on the catalogue M_gas / M*;"
      f" a missing BH is w_jet = 0; zero dust drawn / binned at log = {P9_FLOOR:g}")
_dg_ref = {}
for v in P9_DG_REF:
    h = QM[(QM["box"] == v) & np.isfinite(QM["ldg"])]
    if len(h) >= P9_MIN:
        _dg_ref[v] = (float(h["ldg"].median()), float(h["lfd"].median()), len(h))
        print(f"  {v}: median log M_dust/M_gas {_dg_ref[v][0]:.2f}, log M_dust/M* {_dg_ref[v][1]:.2f} (N = {len(h)}) — the run without the X-ray channel, drawn as the zero-exposure reference")

# (a) the exposure per class: what the classes actually received
print(f"\nthe exposure per class (classified galaxies of QM with SFT / QT; medians):")
print(f"  {'box':8s} {'class':7s} {'N':>4s} {'E_x':>6s} {'f_x':>6s} {'r_x':>6s} {'T_gaspoor':>9s} {'E_jet':>6s} {'dt_SFT':>6s} {'lag_gate':>8s} {'gate@SFT':>8s} {'E_x=0':>6s} {'w_pre':>6s} {'log D/G':>8s} {'log f_d':>8s}")
P9_CLS = {}
for box in P9_BOXES:
    g = QM[(QM["box"] == box) & QM["agn_class"].isin(CLASSES) & np.isfinite(QM["e_x"])]
    P9_CLS[box] = g
    for c in CLASSES + ["all"]:
        h = g if c == "all" else g[g["agn_class"] == c]
        if len(h) < P9_MIN:
            continue
        print(f"  {box:8s} {c:7s} {len(h):4d} {h['e_x'].median():6.2f} {h['f_x'].median():6.2f} {h['r_x'].median():6.2f} {h['t_gaspoor'].median():9.2f} {h['e_jet'].median():6.2f} {h['dt_sft'].median():6.2f} "
              f"{h['lag_gate'].median():8.2f} {100 * h['gate_at_sft'].mean():7.0f}% {100 * (h['e_x'] <= 0).mean():5.0f}% {h['w_pre'].median():6.2f} {h['ldg'].median():8.2f} {h['lfd'].median():8.2f}")

# (a2) the populations the histories define: the first X-ray episode's timing (never / before SFT / in [SFT, QT] / after QT) and the quenching sequence (gas-poor before SFT
#      or star formation stopping first) x class -> medians of the dust content, the disc, the BH, the environment
def pop_row(box, population, h):
    with np.errstate(all="ignore"):
        lmbh = np.log10(np.clip(h["mbh"].to_numpy(float), 1.0, None))
    return dict(box=box, population=population, n=len(h), ldg=h["ldg"].median(), lfd=h["lfd"].median(), nodust=(h["side"] == "no-dust").mean(), dustrich=(h["side"] == "dust-rich").mean(),
                kappa_gas=h["kappa_gas"].median(), age=h["age"].median(), fgas_anchor=h["fgas_anchor"].median(), log_mbh=float(np.median(lmbh)) if len(h) else np.nan, jet=h["jet"].mean(),
                central=h["central"].mean(), w_pre=h["w_pre"].median(), dt_sft=h["dt_sft"].median(), e_x=h["e_x"].median(), log_mstar=h["log_mstar"].median())


POP9 = []
for box in P9_BOXES:
    g = P9_CLS[box]
    if len(g) < 2 * P9_MIN:
        continue
    for k, lab, _, _ in P9_ONSET:
        h = g[g["agn_onset"] == k]
        if len(h) >= P9_MIN:
            POP9.append(pop_row(box, lab, h))
    for gs, lab in [(1.0, "gas-poor before SFT"), (0.0, "star formation stops first")]:
        for c in CLASSES + ["all"]:
            h = g[(g["gate_at_sft"] == gs) & ((g["agn_class"] == c) if c != "all" else True)]
            if len(h) >= P9_MIN:
                POP9.append(pop_row(box, f"{lab}, {c}", h))
    POP9.append(pop_row(box, "all classified", g))
POP9 = pd.DataFrame(POP9)
POP9.to_csv(os.path.join(OUT, "paper_ism_prediction_xray_exposure_populations.csv"), index=False)
print(f"\nthe populations of the histories (medians; the first X-ray episode = the first snapshot with w_jet [f_gas < {XRAY_FGAS_MAX:g}] >= {ONSET_THR:g}; the sequence = f_gas at SFT below / above {XRAY_FGAS_MAX:g}):")
print(f"  {'box':7s} {'population':38s} {'N':>5s} {'log D/G':>8s} {'log f_d':>8s} {'no-dust':>7s} {'dust-rich':>9s} {'kappa_g':>7s} {'age':>5s} {'f_gas':>6s} {'log MBH':>7s} {'jet':>5s} {'central':>7s} {'w_pre':>6s} {'dt_SFT':>6s} {'E_x':>5s} {'log M*':>6s}")
for _, r in POP9.iterrows():
    print(f"  {r['box']:7s} {r['population']:38s} {r['n']:5d} {r['ldg']:8.2f} {r['lfd']:8.2f} {100 * r['nodust']:6.0f}% {100 * r['dustrich']:8.0f}% {r['kappa_gas']:7.2f} {r['age']:5.2f} {r['fgas_anchor']:6.3f} "
          f"{r['log_mbh']:7.2f} {100 * r['jet']:4.0f}% {100 * r['central']:6.0f}% {r['w_pre']:6.2f} {r['dt_sft']:6.2f} {r['e_x']:5.2f} {r['log_mstar']:6.2f}")

# (b) every quantity as a predictor of the dust-to-gas ratio and of the dust fraction: Spearman rho with the continuous targets (zero dust at the floor), the AUC at the two ends
PRED9 = []
for box in P9_BOXES:
    g = P9_CLS[box]
    if len(g) < 2 * P9_MIN:
        continue
    for col, lab, _sl in P9_PRED:
        if col not in g.columns:
            continue
        r = dict(box=box, predictor=col, label=lab, short=_sl, n=int(np.isfinite(g[col].to_numpy(float)).sum()))
        for tcol, tlab in P9_TARGETS:
            ok = np.isfinite(g[col].to_numpy(float)) & np.isfinite(g[tcol].to_numpy(float))
            rho = spearmanr(g.loc[ok, col], g.loc[ok, tcol]) if ok.sum() >= 5 and g.loc[ok, col].nunique() > 1 else None
            r[f"rho_{tcol}"] = float(rho.statistic) if rho is not None else np.nan; r[f"p_{tcol}"] = float(rho.pvalue) if rho is not None else np.nan
            for c in CLASSES:                                                        # the same rank correlation inside each class
                okc = ok & (g["agn_class"] == c).to_numpy()
                rc = spearmanr(g.loc[okc, col], g.loc[okc, tcol]) if okc.sum() >= 5 and g.loc[okc, col].nunique() > 1 else None
                r[f"rho_{tcol}_{c}"] = float(rc.statistic) if rc is not None else np.nan; r[f"p_{tcol}_{c}"] = float(rc.pvalue) if rc is not None else np.nan
        for s, _ in P8_ENDS:
            auc, p, n1, n0 = auc_of(g[col], g["side"] == s)
            r.update({f"auc_{s}": auc, f"p_{s}": p})
        PRED9.append(r)
PRED9 = pd.DataFrame(PRED9)
PRED9.to_csv(os.path.join(OUT, "paper_ism_prediction_xray_exposure_predictors.csv"), index=False)
for box in P9_BOXES:
    pr = PRED9[PRED9["box"] == box]
    if not len(pr):
        continue
    print(f"\n{box}: {len(P9_CLS[box])} classified galaxies with an exposure; Spearman rho with log M_dust/M_gas (all | weak | strong) and with log M_dust/M*, the AUC for dust-rich / no-dust against the rest:")
    print(f"  {'predictor':52s} {'rho(D/G)':>9s} {'p':>7s} {'weak':>6s} {'strong':>6s} {'rho(f_d)':>9s} {'p':>7s} {'AUC rich':>8s} {'AUC none':>8s}")
    for _, r in pr.sort_values("rho_ldg", key=lambda s: -s.abs()).iterrows():
        print(f"  {r['label'][:52]:52s} {r['rho_ldg']:+9.2f} {r['p_ldg']:7.1g} {r['rho_ldg_weak']:+6.2f} {r['rho_ldg_strong']:+6.2f} {r['rho_lfd']:+9.2f} {r['p_lfd']:7.1g} {r['auc_dust-rich']:8.2f} {r['auc_no-dust']:8.2f}")

# (c) the exposure with the class held fixed, and the class with the exposure held fixed: E_x terciles x class (median log D/G, no-dust fraction), Mann–Whitney weak vs strong inside each tercile,
#     Kruskal–Wallis across the terciles inside each class; then the elapsed-time and gas-poor-time controls (the same inside terciles of dt_SFT and of T_gaspoor)
STAT9 = []
for box in P9_BOXES:
    g = P9_CLS[box]
    if len(g) < 3 * P9_MIN:
        continue
    for ctrl, clab in [("e_x", "E_x"), ("dt_sft", "dt_SFT"), ("t_gaspoor", "T_gaspoor")]:
        x = g[ctrl].to_numpy(float); ok = np.isfinite(x)
        q = np.quantile(x[ok], [1 / 3, 2 / 3])
        terc = np.where(~ok, -1, np.where(x <= q[0], 0, np.where(x <= q[1], 1, 2)))
        print(f"\n{box}: {clab} terciles (edges {q[0]:.2f} / {q[1]:.2f} Gyr) x class — median log M_dust/M_gas, no-dust fraction, N; Mann–Whitney p weak vs strong in the tercile"
              + (f"; inside each tercile the rank correlation of E_x with log M_dust/M_gas (what the X-ray weight does at fixed {clab})" if ctrl != "e_x" else ""))
        print(f"  {'tercile':22s} " + " ".join(f"{c:>26s}" for c in CLASSES) + f" {'p weak/strong':>13s}" + (f" {'rho(E_x, D/G)':>14s} {'p':>7s}" if ctrl != "e_x" else ""))
        for k in range(3):
            h = g[terc == k]
            cells, vals = [], {}
            for c in CLASSES:
                hc = h[h["agn_class"] == c]; vals[c] = hc["ldg"].dropna()
                cells.append(f"{hc['ldg'].median():6.2f}  {100 * (hc['side'] == 'no-dust').mean():4.0f} % (n={len(hc):4d})" if len(hc) >= P9_MIN else f"{'—':>26s}")
            p = mannwhitneyu(vals[CLASSES[0]], vals[CLASSES[-1]]).pvalue if all(len(v) >= P9_MIN for v in vals.values()) else np.nan
            row = dict(box=box, control=ctrl, tercile=k, lo=float(x[ok & (terc == k)].min()), hi=float(x[ok & (terc == k)].max()), n=len(h), p_weak_vs_strong=p)
            for c in CLASSES:
                hc = h[h["agn_class"] == c]; row[f"n_{c}"] = len(hc); row[f"ldg_{c}"] = hc["ldg"].median(); row[f"nodust_{c}"] = (hc["side"] == "no-dust").mean()
            extra = ""
            if ctrl != "e_x":
                okk = np.isfinite(h["e_x"].to_numpy(float)) & np.isfinite(h["ldg"].to_numpy(float))
                rr = spearmanr(h.loc[okk, "e_x"], h.loc[okk, "ldg"]) if okk.sum() >= 5 and h.loc[okk, "e_x"].nunique() > 1 else None
                row["rho_ex_ldg"] = float(rr.statistic) if rr is not None else np.nan; row["p_ex_ldg"] = float(rr.pvalue) if rr is not None else np.nan
                extra = f" {row['rho_ex_ldg']:+14.2f} {row['p_ex_ldg']:7.1g}"
            STAT9.append(row)
            print(f"  {k + 1} [{row['lo']:.2f}, {row['hi']:.2f}]".ljust(23) + " ".join(f"{s:>26s}" for s in cells) + f" {p:13.2g}" + extra)
        if ctrl == "e_x":
            for c in CLASSES:
                hs = [g.loc[(terc == k) & (g["agn_class"] == c), "ldg"].dropna() for k in range(3)]
                if all(len(v) >= P9_MIN for v in hs):
                    print(f"  {c:7s}: Kruskal–Wallis across the E_x terciles p = {kruskal(*hs).pvalue:.2g} (medians " + " / ".join(f"{v.median():.2f}" for v in hs) + ")")
STAT9 = pd.DataFrame(STAT9)
STAT9.to_csv(os.path.join(OUT, "paper_ism_prediction_xray_exposure_stats.csv"), index=False)

# (d) the contrasts: the never-exposed galaxies against the exposed ones, and inside each class the two quenching sequences
_VARS9 = tuple(v for v in _VARS8 if v[0] not in ("lfgas",)) + (("lfgas", "log M_gas/M*"), ("ldg", "log M_dust/M_gas"), ("e_x", "E_x [Gyr]"), ("f_x", "f_x"), ("r_x", "r_x"),
                                                               ("t_gaspoor", "T_gaspoor [Gyr]"), ("dt_sft", "dt_SFT [Gyr]"), ("lag_gate", "lag_gate [Gyr]"), ("e_x_pre", "E_x pre-SFT"), ("fgas_sft", "f_gas at SFT"))
for box in P9_BOXES:
    g = P9_CLS[box]
    grp = {"exposed": g[g["agn_onset"] != "never"], "never exposed": g[g["agn_onset"] == "never"]}
    if all(len(v) >= P9_MIN for v in grp.values()):
        contrast_table(f"{box}, the galaxies never exposed to the X-ray channel against the exposed ones", grp, "exposed", cols=_VARS9)
    else:
        print(f"\n{box}: {len(grp['never exposed'])} never-exposed galaxies — no contrast")
    for c in CLASSES:
        grp = {f"{c}, gas-poor first": g[(g["agn_class"] == c) & (g["gate_at_sft"] == 1.0)], f"{c}, SF stops first": g[(g["agn_class"] == c) & (g["gate_at_sft"] == 0.0)]}
        if all(len(v) >= P9_MIN for v in grp.values()):
            contrast_table(f"{box}, {c} coupling: the quenching sequence (f_gas at SFT below / above {XRAY_FGAS_MAX:g})", grp, f"{c}, gas-poor first", cols=_VARS9)


# (e) the figure: per box a column; (a) log M_dust/M_gas against E_x, every galaxy + the per-class binned medians, the no-X-ray runs' medians as the zero-exposure reference;
#     (b) log M_dust/M_gas against the gas-poor time T_gaspoor, the galaxies split by r_x (the jet weight while gas-poor) terciles — the gate held fixed; (c) the rank correlations
def bin_median(x, y, edges, nmin=P9_MIN, n=NBOOT, seed=0):
    """Median of y in the fixed x bins `edges` (the first bin = x == edges[0] exactly when edges[0] == edges[1]) with a bootstrap 16–84 % band -> DataFrame (x, y, lo, hi, n)."""
    x, y = np.asarray(x, float), np.asarray(y, float); ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    rng, out = np.random.default_rng(seed), []
    for i in range(len(edges) - 1):
        m = (x == edges[0]) if (i == 0 and edges[0] == edges[1]) else ((x > edges[i]) & (x <= edges[i + 1]))
        if m.sum() < nmin:
            continue
        yy = y[m]; bs = np.median(yy[rng.integers(0, len(yy), (n, len(yy)))], axis=1)
        out.append((float(np.median(x[m])), float(np.median(yy)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), int(m.sum())))
    return pd.DataFrame(out, columns=["x", "y", "lo", "hi", "n"])


if len(P9_BOXES):
    _nb = len(P9_BOXES)
    fig, axes = plt.subplots(4, _nb, figsize=(5.2 * _nb + 0.8, 17.6), squeeze=False, gridspec_kw=dict(hspace=0.36, wspace=0.22))
    _RX_COL = {0: "#67a9cf", 1: "#ef8a62", 2: "#b2182b"}
    _YL = (P9_FLOOR - 0.3, -0.2)
    _rows9 = []
    for i, box in enumerate(P9_BOXES):
        g = P9_CLS[box]
        # (a) D/G against the exposure, per class; the E_x = 0 pile is its own bin
        ax = axes[0][i]
        _xj = g["e_x"].to_numpy(float) + np.where(g["e_x"].to_numpy(float) <= 0, np.random.default_rng(1).uniform(-0.04, 0.04, len(g)), 0.0)
        for c in CLASSES:
            m = (g["agn_class"] == c).to_numpy()
            ax.scatter(_xj[m], g.loc[m, "ldg"], s=9, color=CLASS_COLOR[c], alpha=0.35, lw=0, zorder=2)
            tr = bin_median(g.loc[m, "e_x"], g.loc[m, "ldg"], P9_EX_EDGES)
            if len(tr):
                ax.fill_between(tr["x"], tr["lo"], tr["hi"], color=CLASS_COLOR[c], alpha=0.16, lw=0, zorder=3)
                ax.plot(tr["x"], tr["y"], color=CLASS_COLOR[c], lw=2.6, marker="o", ms=6, mec="k", mew=0.5, zorder=4, path_effects=[withStroke(linewidth=4.2, foreground="white")],
                        label=f"{c} AGN coupling (N = {int(m.sum())})")
                for _, r in tr.iterrows():
                    _rows9.append(dict(box=box, panel="ldg_vs_ex", group=c, x=r["x"], y=r["y"], lo=r["lo"], hi=r["hi"], n=r["n"]))
        tr = bin_median(g["e_x"], g["ldg"], P9_EX_EDGES)
        if len(tr):
            ax.plot(tr["x"], tr["y"], color="0.15", lw=2.0, ls=(0, (1.2, 1.4)), marker="s", ms=5, zorder=5, label=f"all classified (N = {len(g)})")
            for _, r in tr.iterrows():
                _rows9.append(dict(box=box, panel="ldg_vs_ex", group="all", x=r["x"], y=r["y"], lo=r["lo"], hi=r["hi"], n=r["n"]))
        for v, (dg, fd, nv) in _dg_ref.items():
            ax.axhline(dg, color=BOXES[v]["color"], lw=1.6, ls=(0, (4.0, 1.6)), zorder=1, label=f"{P5_SHORT.get(v, v)} median (no X-ray channel, N = {nv})")
        pr = PRED9[(PRED9["box"] == box) & (PRED9["predictor"] == "e_x")]
        if len(pr):
            r = pr.iloc[0]
            ax.text(0.98, 0.97, rf"$\rho$($E_x$, D/G) = {r['rho_ldg']:+.2f} (p = {r['p_ldg']:.1g}); weak {r['rho_ldg_weak']:+.2f}, strong {r['rho_ldg_strong']:+.2f}",
                    transform=ax.transAxes, ha="right", va="top", fontsize=FONT["note"] + 0.3, color="0.25", zorder=6)
        ax.axhline(P9_FLOOR + 0.25, color="0.6", lw=0.8, ls=":", zorder=1); ax.text(0.98, 0.035, "zero dust (floor)", transform=ax.transAxes, fontsize=FONT["note"] - 0.5, color="0.45", ha="right", va="bottom")
        ax.set_xlim(-0.15, min(6.0, max(0.6, float(np.nanmax(g["e_x"])) * 1.08))); ax.set_ylim(*_YL)
        ax.set_ylabel(r"$\log\,M_{\rm dust}\,/\,M_{\rm gas}$" if i == 0 else ""); ax.set_xlabel(r"X-ray exposure $E_x = \int_{\rm SFT}^{\rm anchor} w_{\rm jet}\,[f_{\rm gas} < 0.2]\,{\rm d}t$  [Gyr]")
        ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        ax.set_title(f"({'abc'[i]}) {BOXES[box]['label'].split(' (')[0]}: dust-to-gas against the exposure", loc="left", fontsize=FONT["tag"])
        ax.legend(loc="lower left", frameon=False, fontsize=FONT["legend"] - 1.5)
        # (b) D/G against the time since SFT, the galaxies grouped by the timing of their first X-ray episode
        ax = axes[1][i]
        _dt = g["dt_sft"].to_numpy(float); _ted = list(np.quantile(_dt[np.isfinite(_dt)], np.linspace(0, 1, 7)))
        for k, lab, col, mk in P9_ONSET:
            m = (g["agn_onset"] == k).to_numpy()
            if m.sum() < P9_MIN:
                continue
            ax.scatter(_dt[m], g.loc[m, "ldg"], s=9 if k != "never" else 16, color=col, alpha=0.35 if k != "never" else 0.8, lw=0, zorder=2 if k != "never" else 5)
            tr = bin_median(_dt[m], g.loc[m, "ldg"].to_numpy(float), _ted, nmin=P9_MIN)
            if len(tr):
                if k != "never":
                    ax.fill_between(tr["x"], tr["lo"], tr["hi"], color=col, alpha=0.16, lw=0, zorder=3)
                ax.plot(tr["x"], tr["y"], color=col, lw=2.6, marker=mk, ms=6, mec="k", mew=0.5, zorder=4 if k != "never" else 6, path_effects=[withStroke(linewidth=4.2, foreground="white")],
                        label=f"{lab} (N = {int(m.sum())})")
                for _, r in tr.iterrows():
                    _rows9.append(dict(box=box, panel="ldg_vs_dtsft", group=k, x=r["x"], y=r["y"], lo=r["lo"], hi=r["hi"], n=r["n"]))
            elif k == "never":
                ax.plot([np.nanmedian(_dt[m])], [np.nanmedian(g.loc[m, "ldg"])], color=col, ls="", marker=mk, ms=9, mec="k", mew=0.6, zorder=6, label=f"{lab} (N = {int(m.sum())}, one median)")
        for v, (dg, fd, nv) in _dg_ref.items():
            ax.axhline(dg, color=BOXES[v]["color"], lw=1.6, ls=(0, (4.0, 1.6)), zorder=1)
        ax.set_ylim(*_YL); ax.set_xlim(-0.15, min(8.0, max(0.6, float(np.nanmax(_dt)) * 1.08)))
        ax.set_ylabel(r"$\log\,M_{\rm dust}\,/\,M_{\rm gas}$" if i == 0 else ""); ax.set_xlabel(r"time since SFT, $t_{\rm anchor} - t_{\rm SFT}$  [Gyr]")
        ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        ax.set_title(f"({'def'[i]}) the timing of the first X-ray episode", loc="left", fontsize=FONT["tag"])
        ax.legend(loc="lower left", frameon=False, fontsize=FONT["legend"] - 1.5)
        # (c) D/G against the gas-poor time, split by the jet weight while gas-poor: the gate held fixed
        ax = axes[2][i]
        rx = g["r_x"].to_numpy(float); okr = np.isfinite(rx)
        hh = g[~okr]
        if len(hh):
            ax.scatter(hh["t_gaspoor"], hh["ldg"], s=9, color="0.7", alpha=0.5, lw=0, zorder=2, label=f"never gas-poor after SFT (N = {len(hh)})")
        if okr.sum() >= 3 * P9_MIN:
            qq = np.quantile(rx[okr], [1 / 3, 2 / 3]); terc = np.where(rx <= qq[0], 0, np.where(rx <= qq[1], 1, 2))
            _tg = g["t_gaspoor"].to_numpy(float); _ted = list(np.quantile(_tg[okr], np.linspace(0, 1, 6)))
            for k in range(3):
                m = okr & (terc == k)
                if m.sum() < P9_MIN:
                    continue
                lab = [f"$r_x \\leq$ {qq[0]:.2f}", f"{qq[0]:.2f} < $r_x \\leq$ {qq[1]:.2f}", f"$r_x >$ {qq[1]:.2f}"][k]
                ax.scatter(_tg[m], g.loc[m, "ldg"], s=9, color=_RX_COL[k], alpha=0.35, lw=0, zorder=2)
                tr = bin_median(_tg[m], g.loc[m, "ldg"].to_numpy(float), _ted)
                if len(tr):
                    ax.fill_between(tr["x"], tr["lo"], tr["hi"], color=_RX_COL[k], alpha=0.16, lw=0, zorder=3)
                    ax.plot(tr["x"], tr["y"], color=_RX_COL[k], lw=2.6, marker="o", ms=6, mec="k", mew=0.5, zorder=4, path_effects=[withStroke(linewidth=4.2, foreground="white")],
                            label=f"jet weight while gas-poor {lab} (N = {int(m.sum())})")
                    for _, r in tr.iterrows():
                        _rows9.append(dict(box=box, panel="ldg_vs_tgaspoor", group=f"r_x tercile {k + 1}", x=r["x"], y=r["y"], lo=r["lo"], hi=r["hi"], n=r["n"]))
        st = STAT9[(STAT9["box"] == box) & (STAT9["control"] == "t_gaspoor")]
        if len(st):
            ax.text(0.98, 0.97, r"$\rho$($E_x$, D/G) inside the $T_{\rm gas-poor}$ terciles: " + ", ".join(f"{r['rho_ex_ldg']:+.2f}" for _, r in st.iterrows()),
                    transform=ax.transAxes, ha="right", va="top", fontsize=FONT["note"] + 0.3, color="0.25", zorder=6)
        ax.set_ylim(*_YL); ax.set_xlim(-0.15, min(6.0, max(0.6, float(np.nanmax(g["t_gaspoor"])) * 1.08)))
        ax.set_ylabel(r"$\log\,M_{\rm dust}\,/\,M_{\rm gas}$" if i == 0 else ""); ax.set_xlabel(r"gas-poor time $T_{\rm gas-poor} = \int_{\rm SFT}^{\rm anchor} [f_{\rm gas} < 0.2]\,{\rm d}t$  [Gyr]")
        ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        ax.set_title(f"({'ghi'[i]}) the gate held fixed: the jet weight while gas-poor", loc="left", fontsize=FONT["tag"])
        ax.legend(loc="lower left", frameon=False, fontsize=FONT["legend"] - 1.5)
        # (d) the rank correlations of every predictor with log D/G and log f_dust
        ax = axes[3][i]
        _cols = [c for c, _, _ in P9_PRED if c in PRED9["predictor"].values]
        pr = PRED9[PRED9["box"] == box].set_index("predictor").reindex(_cols)
        yy = np.arange(len(pr))[::-1]
        ax.barh(yy + 0.18, pr["rho_ldg"], height=0.34, color="#7b3294", label=r"Spearman $\rho$ with log $M_{\rm dust}/M_{\rm gas}$", zorder=3)
        ax.barh(yy - 0.18, pr["rho_lfd"], height=0.34, color="#c2a5cf", label=r"Spearman $\rho$ with log $M_{\rm dust}/M_\star$", zorder=3)
        for c in CLASSES:
            ax.plot(pr[f"rho_ldg_{c}"], yy + 0.18, ls="", marker="|", ms=9, mew=1.8, color=CLASS_COLOR[c], zorder=4, label=f"the same inside the {c} class (D/G)")
        ax.axvline(0, color="0.3", lw=1.0); ax.set_yticks(yy); ax.set_yticklabels(list(pr["short"]) if i == 0 else [""] * len(yy), fontsize=FONT["tick"] - 0.5)
        ax.set_xlim(-0.75, 0.75); ax.set_xlabel(r"Spearman $\rho$ (zero dust at the floor)"); ax.tick_params(direction="in", top=True, right=True, labelsize=FONT["tick"]); ax.grid(False)
        ax.set_title(f"({'jkl'[i]}) every quantity against the dust content", loc="left", fontsize=FONT["tag"])
        if i == 0:
            _hd, _lb = ax.get_legend_handles_labels()
    fig.legend(_hd, _lb, loc="lower center", ncol=4, frameon=False, fontsize=FONT["legend"], bbox_to_anchor=(0.5, 0.005))
    fig.suptitle(f"the X-ray channel after quenching: exposure $E_x$ = the jet weight while $f_{{\\rm gas}} < {XRAY_FGAS_MAX:g}$, integrated from SFT to the anchor (catalogue histories);"
                 f" quenched, {SEL_TXT.replace(chr(10), ' ')}, classified galaxies", fontsize=FONT["tag"], y=0.995)
    fig.subplots_adjust(bottom=0.055)
    paper_save(fig, "paper_ism_prediction_xray_exposure")
    plt.show()
    pd.DataFrame(_rows9).to_csv(os.path.join(OUT, "paper_ism_prediction_xray_exposure.csv"), index=False)
    print(f"  tables -> {os.path.join(OUT, 'paper_ism_prediction_xray_exposure{.csv,_predictors.csv,_stats.csv,_populations.csv}')}, {EXP_CSV}")

## Conventions, caveats, what the pieces are

* **What is identical to figure H** — the panel grid, axis limits, the observed points (ALMA-C11 fiducial CIGALE values with their censors, Spilker+18 LEGA-C with DR3 structure / SED, ADF22-QG1), the $A_V$ colour scale, the mass cut, the selection *rule* (m25 Part 1 constants), the `obs_vs_track` bookkeeping.
* **What is different** — the model side is a catalogue column: whole-galaxy (FOF) dust, H$_2$, stellar mass, instantaneous `sfr` and mass-weighted age, one row per galaxy (no sightlines); $R_{\rm e}$ is $0.75\times$ the 3-D half-mass radius (`R_PROJ_OVER_3D`; the m25 value is the projected half-mass radius of the curve of growth). The m25 reference drawn here is therefore the m25 particle sample *read from the cis25 catalogue* (`M25_REF_MODEL = "catalogue"`: the Q galaxies of the selection table, the same columns as every box, its own classes), so an offset between a box and the reference is resolution / volume (m25 vs m100 / m50) and nothing else. No particle product is that galaxy: on the 266 Q the projected 0–10 kpc disc holds +0.26 dex more dust and +0.34 dex more gas than the FOF galaxy (a disc is a cylinder through the 100 kpc cut-out), the 0–100 kpc rung +1.3 dex, and the H$_2$ of `annulus_ism_truth` is still the pre-fix recipe (−0.6 dex). The particle references stay as options — `"sim"` (figure H$_{\rm sim}$, the truth in the 0–10 / 0–32 kpc discs) and `"cigale"` (figure H proper, the CIGALE fits of the mock *core* photometry: aperture and estimator mixed into the offset, a core being nothing a catalogue can reproduce). The `cis25` catalogue track is the m25 box under the catalogue rule: against the reference it isolates the selection (particle vs catalogue rule), against the larger boxes the resolution.
* **Resolution enters through the selection** — the $\geq 21$ gas particle cut of the rule is a gas mass floor of $4\times10^8$ M$_\odot$ in the 100 / 50 Mpc boxes against $5\times10^7$ M$_\odot$ in the 25 Mpc box; the quenched galaxies of the large boxes that survive it are the gas-richer end of the passive pool (the funnel of Part 1 shows how many the cut removes). The low-redshift passive pool of the large boxes is satellite-dominated (`CENTRALS_ONLY`).
* **The 50 Mpc box and its variants** — the share holds the ten anchor catalogues of `s50` (`SIMBA_50/Groups`), `s50nox` and `s50noagn` (`SIMBA_50/<variant>/Groups`, fetched from ROE on 2026-08-30); `cis50nojet` is configured but not downloaded (`ROE_CATALOG_URL`, `wget -c`). The `AVAIL` printout of Part 0 lists what is missing; a re-run reads only the new (box, anchor) pairs into the cache.
* **The feedback variants (Part 4)** — s50nox / s50nojet / s50noagn are independent runs of the same initial conditions: their quenched populations differ in size and nature (without jets few massive galaxies quench at all), so the variant tracks compare *the galaxies that quench under that physics*, not the same galaxies re-simulated. Their catalogues are not on the share by default (`SIMBA_50/<variant>/Groups`, ROE `m50n512/<variant>/catalogs`, ~200 MB each).
* **The AGN coupling classes (Part 6)** — the m25 `pre_threshold` rule re-implemented on the catalogue histories (main progenitor branch from the in-catalogue trees; the cis25 sidecar trees of the m25 pipeline), with the same `find_quenching_times`, constants and thresholds; the cis25 labels are checked galaxy by galaxy against the m25 selection table. An anchor is classified only when its whole catalogue ladder (`END_SNAP[anchor]` … anchor) is on disk: for the 50 Mpc box that is the $z \geq 1.15$ anchors until snapshots 101–133 are fetched (ROE), for `s50nox` nothing until its ladder is fetched, and `s50noagn` has no trees at all (and no AGN feedback to couple). Parts 3 and 5 use the anchor-epoch state instead (in m25 the anchor state was nearly uninformative: most quenched galaxies are in jet mode by then).
* **The AGN state across the variants (Part 5)** — the jet criterion is evaluated on every run's own catalogue, so a state is the same $M_{\rm BH}$ / $f_{\rm Edd}$ selection under different physics: in `s50nox` a jet-mode galaxy has the jets without the X-ray heating, in `s50noagn` the label only says its BH grew. Comparing one state across the variants therefore reads what the missing channel does to galaxies in that state; comparing the states inside the fiducial run reads what not (yet) having a grown BH does. Both are anchor-epoch statements: the histories (dose, timing) are not in the catalogues.
* Zero dust / H$_2$ rows are dropped on the log axes (never floored); zero SFR is drawn at the sSFR floor. The cache stores every galaxy above `POOL_LOGM`, so the cuts can be varied without re-reading the catalogues.
* **The class scheme** — `CLASS_SCHEME = "two"` collapses the m25 rule to weak / strong (intermediate redistributed at `TWO_CLASS_THR` = 0.30 on $w_{\rm pre}$) everywhere the classes are used: Part 6, the catalogue reference of Part 2, the Part 7 populations. The rule's three-class label survives as `agn_class3` (and in the class CSV), so the check against the m25 pipeline is unaffected; the `sim` / `cigale` reference CSVs carry the rule's three classes (only their weak / strong tracks are drawn).
* **The dusty tail (Part 7)** — the tracks are medians; the galaxies that reach the observed $M_{\rm dust}/M_\star$ are a minority of every fiducial box (a few per cent above $-3.0$, $\sim 10$ % above $-3.2$) and the majority of the no-X-ray / no-AGN runs. Part 7 tabulates them per population, contrasts them with the rest and draws them one by one with the running 84th / 95th percentiles. In the fiducial boxes the tail is not the upper end of a continuous scatter but a **separate cloud** at $M_{\rm dust}/M_\star \approx 10^{-3}$ — the ALMA-C11 detections' locus — with a nearly empty gap down to $10^{-3.5}$: a quenched SIMBA galaxy either keeps its dust at a normal dust-to-gas ratio (younger, gas-richer, more often a satellite, less often in jet mode) or has lost it wholesale; the running median follows the majority and never crosses the gap.
* **The three dust sides (Part 8)** — the cuts at $M_{\rm dust}/M_\star = 10^{-5}$ and $10^{-3.5}$ (`P8_EDGES`: dust-rich, undetected, no-dust) make the two-sidedness a label and ask whether the pre-quenching AGN coupling class predicts it: the side fractions per class with Wilson intervals, the $\chi^2$ $p$ and Cramér's $V$ of the class $\times$ side table, the best class $\Rightarrow$ side rule against the majority baseline, Fisher's exact test and the weak-vs-strong odds at each end, and the AUC of $w_{\rm pre}$ next to that of the cheap quantities (age, sSFR, gas fraction, jet flag, satellite flag …) for dust-rich against the rest and for no-dust against the rest. The per-box grids draw the side $\times$ class medians; the H$_2$ row is where a class difference at fixed side would show.
* **The X-ray exposure (Part 9)** — the same histories give what the channel did *after* quenching: $E_x = \int w_{\rm jet}[f_{\rm gas} < 0.2]\,{\rm d}t$ from SFT to the anchor, the gas-poor time, the jet weight while gas-poor, the first episode's timing and the quenching sequence ($f_{\rm gas}$ at SFT). The target is the dust-to-gas ratio (the gate is a gas cut). The channel reaches 98–100 % of the classified quenched galaxies and saturates within $\sim 0.5$ Gyr (D/G from $-1.4$ to $-3.5$); the never-exposed 1.3 % of cis100 (undergrown BH, satellites) are the s50nox phenotype at the observed $M_{\rm dust}/M_\star$; among the exposed the timing of the first episode and the sequence (gas-poor before SFT keeps 0.5 dex more D/G) set the dust content, the class only modulates it (strong keeps 0.15–0.5 dex more at fixed exposure). Whole-galaxy $f_{\rm gas} = M_{\rm gas}/M_\star$ of the catalogue stands in for the on-the-fly gas fraction of the code; snapshots ~100 Myr apart; a missing BH is $w_{\rm jet} = 0$.
